# QSPR/QSAR workflow for `Log_S1` prediction

This notebook contains the full QSPR/QSAR workflow used for modeling `Log_S1` in a small heterogeneous molecular dataset comprising bimetallic clusters, individual bases, and bimetal–base complexes.

The analysis includes descriptor loading and validation, train-only preprocessing, fixed external test assignment, correlation and VIF diagnostics, repeated complex-only cross-validation, feature-subset comparison, model optimization, applicability-domain analysis, Y-randomization, robustness assessment, and model interpretation.

The external test set is excluded from preprocessing, feature selection, hyperparameter optimization, and internal cross-validation. External-test metrics are calculated only after model fitting and are subsequently included in the comparative model assessment.

The workflow uses the descriptor list provided in `62_descriptors.txt` and a fixed set of external-test `Molecule_ID` values to ensure reproducibility across runs.


In [1]:
# ============================================================
# Analysis setup, reproducibility, and utilities
# ============================================================

import os
import re
import json
import math
import time
import copy
import random
import shutil
import warnings
import platform
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import spearmanr

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LassoCV
from sklearn.cross_decomposition import PLSRegression
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.tree import DecisionTreeRegressor

import joblib

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

try:
    import optuna

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
except Exception:
    optuna = None
    HAS_OPTUNA = False

try:
    import shap

    HAS_SHAP = True
except Exception:
    shap = None
    HAS_SHAP = False

try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor

    HAS_STATSMODELS = True
except Exception:
    variance_inflation_factor = None
    HAS_STATSMODELS = False

try:
    from xgboost import XGBRegressor

    HAS_XGBOOST = True
except Exception:
    XGBRegressor = None
    HAS_XGBOOST = False

try:
    from lightgbm import LGBMRegressor

    HAS_LIGHTGBM = True
except Exception:
    LGBMRegressor = None
    HAS_LIGHTGBM = False

try:
    from catboost import CatBoostRegressor

    HAS_CATBOOST = True
except Exception:
    CatBoostRegressor = None
    HAS_CATBOOST = False

try:
    from lazypredict.Supervised import LazyRegressor

    HAS_LAZYPREDICT = True
except Exception:
    LazyRegressor = None
    HAS_LAZYPREDICT = False

try:
    import importlib.metadata as importlib_metadata
except Exception:
    import importlib_metadata

# -----------------------------
# Analysis settings
# -----------------------------

RANDOM_STATE = 42

DO_CORRELATION_FILTERING = True
CORRELATION_CUTOFF = 0.95

DO_VIF_FILTERING = True
VIF_CUTOFF = 10.0

TOP_N_MODELS = 3

RUN_MODE = "hourly"
OVERWRITE_EXISTING_RESULTS = False

FAST_MODE = False

N_OPTUNA_TRIALS = 40
N_REPEATS_CV = 20
N_SPLITS_CV = 5
N_Y_RANDOMIZATION = 300
N_BOOTSTRAP = 500
N_MONTE_CARLO_SPLITS = 300

# Y-randomization is restricted to preliminary candidates to limit computational cost.
# Preliminary candidates are selected using internal CV and external-test metrics,
# followed by applicability-domain and Y-randomization criteria in the final ranking.
MAX_MODELS_FOR_ROBUST_RANKING = 10

if FAST_MODE:
    N_OPTUNA_TRIALS = 8
    N_REPEATS_CV = 5
    N_SPLITS_CV = 5
    N_Y_RANDOMIZATION = 30
    N_BOOTSTRAP = 50
    N_MONTE_CARLO_SPLITS = 50
    MAX_MODELS_FOR_ROBUST_RANKING = 6

N_OPTUNA_CV_REPEATS = min(5, N_REPEATS_CV)
N_NESTED_OUTER_REPEATS = 2 if not FAST_MODE else 1
N_NESTED_OPTUNA_TRIALS = min(12, N_OPTUNA_TRIALS) if not FAST_MODE else 5
N_ALE_BINS = 10
N_TOP_FEATURES_FOR_INTERPRETATION = 6
N_SHAP_MAX_SAMPLES = 100
N_PERMUTATION_REPEATS = 50 if not FAST_MODE else 10

DATA_FILE = Path("All_molecules_noncorelated_descriptors_and_y_quantum.xlsx")
DESCRIPTOR_FILE = Path("62_descriptors_and_quantum.txt")
TEST_FILE = Path("test_molecules.csv")

TARGET_COLUMN_CANDIDATES = ["Log_S1", "logS1", "LogS1", "log_s1", "LOG_S1"]

EXTERNAL_TEST_IDS = [
    50, 31, 65, 81, 8, 35, 49, 76, 18, 54,
    92, 44, 66, 97, 20, 12, 27, 37, 59, 72
]

REQUIRED_SHEETS = [
    "raw_file_audit", "descriptor_mapping", "missing_descriptors",
    "molecule_type_assignment", "train_test_split", "preprocessing_report",
    "correlation_pairs", "high_correlation_pairs", "vif_before", "high_vif",
    "feature_selection_lasso", "feature_selection_rfe", "feature_selection_permutation",
    "feature_selection_shap", "feature_rank_aggregation", "feature_subset_comparison",
    "model_specific_best_subsets", "all_model_metrics", "model_ranking",
    "top3_model_metrics", "top3_predictions", "applicability_domain",
    "y_randomization", "bootstrap_validation", "monte_carlo_validation",
    "final_selected_features", "ensemble_predictions", "report_summary"
]

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.grid": True,
    "grid.alpha": 0.25,
})


def package_version(pkg_name: str) -> str:
    try:
        return importlib_metadata.version(pkg_name)
    except Exception:
        return "not installed"


def make_results_dir(run_mode="hourly") -> Path:
    now = datetime.now()
    if run_mode == "daily":
        stamp = now.strftime("%Y%m%d")
    else:
        stamp = now.strftime("%Y%m%d_%H")
    outdir = Path(f"results_QSPR_QSAR_LogS1_{stamp}")
    outdir.mkdir(parents=True, exist_ok=True)
    return outdir


RESULTS_DIR = make_results_dir(RUN_MODE)
FIG_DIR = RESULTS_DIR / "figures"
MODEL_DIR = RESULTS_DIR / "models"
TABLE_DIR = RESULTS_DIR / "tables"
REPORT_DIR = RESULTS_DIR / "report"

for d in [FIG_DIR, MODEL_DIR, TABLE_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def versioned_path(path: Path, overwrite: bool = False) -> Path:
    path = Path(path)
    if overwrite or not path.exists():
        return path
    parent, stem, suffix = path.parent, path.stem, path.suffix
    version = 2
    while True:
        candidate = parent / f"{stem}_v{version}{suffix}"
        if not candidate.exists():
            return candidate
        version += 1


def safe_sheet_name(name: str) -> str:
    name = str(name).replace("/", "_").replace("\\", "_").replace("*", "_")
    name = name.replace("?", "_").replace("[", "_").replace("]", "_").replace(":", "_")
    return name[:31]


def save_dataframe(df: pd.DataFrame, filename: str, index: bool = False) -> Path:
    path = versioned_path(TABLE_DIR / filename, OVERWRITE_EXISTING_RESULTS)
    if path.suffix.lower() == ".xlsx":
        df.to_excel(path, index=index)
    elif path.suffix.lower() == ".csv":
        df.to_csv(path, index=index)
    else:
        df.to_excel(path.with_suffix(".xlsx"), index=index)
    return path


def save_figure(fig, name: str):
    clean_name = re.sub(r"[^A-Za-z0-9_\-\.]+", "_", str(name))[:180]
    png_path = versioned_path(FIG_DIR / f"{clean_name}.png", OVERWRITE_EXISTING_RESULTS)
    pdf_path = versioned_path(FIG_DIR / f"{clean_name}.pdf", OVERWRITE_EXISTING_RESULTS)
    fig.tight_layout()
    fig.savefig(png_path, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)
    return png_path, pdf_path


TABLES = {sheet: pd.DataFrame() for sheet in REQUIRED_SHEETS}
WARNINGS = []


def add_warning(message: str):
    WARNINGS.append(message)
    print(f"WARNING: {message}")


RUN_METADATA = {
    "datetime": datetime.now().isoformat(),
    "random_state": RANDOM_STATE,
    "run_mode": RUN_MODE,
    "results_dir": str(RESULTS_DIR),
    "fast_mode": FAST_MODE,
    "external_test_ids": EXTERNAL_TEST_IDS,
    "settings": {
        "DO_CORRELATION_FILTERING": DO_CORRELATION_FILTERING,
        "CORRELATION_CUTOFF": CORRELATION_CUTOFF,
        "DO_VIF_FILTERING": DO_VIF_FILTERING,
        "VIF_CUTOFF": VIF_CUTOFF,
        "TOP_N_MODELS": TOP_N_MODELS,
        "N_OPTUNA_TRIALS": N_OPTUNA_TRIALS,
        "N_REPEATS_CV": N_REPEATS_CV,
        "N_SPLITS_CV": N_SPLITS_CV,
        "N_Y_RANDOMIZATION": N_Y_RANDOMIZATION,
        "N_BOOTSTRAP": N_BOOTSTRAP,
        "N_MONTE_CARLO_SPLITS": N_MONTE_CARLO_SPLITS,
        "MAX_MODELS_FOR_ROBUST_RANKING": MAX_MODELS_FOR_ROBUST_RANKING
    },
    "python": platform.python_version(),
    "platform": platform.platform(),
    "packages": {
        "numpy": package_version("numpy"),
        "pandas": package_version("pandas"),
        "scipy": package_version("scipy"),
        "scikit-learn": package_version("scikit-learn"),
        "matplotlib": package_version("matplotlib"),
        "seaborn": package_version("seaborn"),
        "optuna": package_version("optuna"),
        "shap": package_version("shap"),
        "xgboost": package_version("xgboost"),
        "lightgbm": package_version("lightgbm"),
        "catboost": package_version("catboost"),
        "lazypredict": package_version("lazypredict"),
        "statsmodels": package_version("statsmodels"),
        "joblib": package_version("joblib"),
        "openpyxl": package_version("openpyxl")
    }
}

metadata_path = versioned_path(RESULTS_DIR / "run_metadata.json", OVERWRITE_EXISTING_RESULTS)
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(RUN_METADATA, f, ensure_ascii=False, indent=2)

print(f"Results directory: {RESULTS_DIR.resolve()}")
print(
    f"Optuna: {HAS_OPTUNA}; SHAP: {HAS_SHAP}; XGBoost: {HAS_XGBOOST}; LightGBM: {HAS_LIGHTGBM}; CatBoost: {HAS_CATBOOST}; LazyPredict: {HAS_LAZYPREDICT}")


Results directory: C:\Users\plato\PycharmProjects\DNA\results_QSPR_QSAR_LogS1_20260702_13
Optuna: True; SHAP: True; XGBoost: True; LightGBM: True; CatBoost: True; LazyPredict: True


## 1. Data loading and input audit

The main molecular dataset and the descriptor list are checked before the analysis starts. The Excel table is loaded into `raw_df`, while basic file information is stored in `raw_file_audit` for reproducibility.

Required inputs:

- `All_molecules_noncorelated_descriptors_and_y.xlsx` — molecular data, descriptors, and target values;
- `62_descriptors.txt` — final GA-selected descriptor names;
- `test_molecules.csv` — optional reference file containing the predefined external-test molecules.

If the descriptor list is unavailable, execution is stopped rather than substituting descriptors from another source.


In [2]:
# ============================================================
# Input data loading and file audit
# ============================================================

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Main data file was not found: {DATA_FILE}. "
        "Place All_molecules_noncorelated_descriptors_and_y.xlsx in the working directory."
    )

if not DESCRIPTOR_FILE.exists():
    raise FileNotFoundError(
        f"Descriptor list file was not found: {DESCRIPTOR_FILE}. "
        "The notebook must read descriptor names strictly from 62_descriptors.txt. No Excel fallback is allowed."
    )

raw_df = pd.read_excel(DATA_FILE)

raw_file_audit_records = []
for file_path in [DATA_FILE, DESCRIPTOR_FILE, TEST_FILE]:
    raw_file_audit_records.append({
        "file": str(file_path),
        "exists": file_path.exists(),
        "size_bytes": file_path.stat().st_size if file_path.exists() else np.nan,
        "absolute_path": str(file_path.resolve()) if file_path.exists() else "not found"
    })

TABLES["raw_file_audit"] = pd.DataFrame(raw_file_audit_records)
print(f"Loaded data shape: {raw_df.shape}")
display(raw_df.head())
display(TABLES["raw_file_audit"])


Loaded data shape: (100, 847)


,SrNo,ALogPS_logP,ALogPS_logS,MW:(alvaDesc),AMW:(alvaDesc),Sv:(alvaDesc),Se:(alvaDesc),Sp:(alvaDesc),Mv:(alvaDesc),Mp:(alvaDesc),...,E(LUMO),HOMO_LUMO_gap,Koopmans_IP,Koopmans_EA,Electronegativity,Chemical_hardness,Chemical_softness,Electrophilicity_index,Dipole_moment,Log_S1
0,1,-1.30,-0.15,215.70,107.90,2.07,1.33,8.18,1.04,4.09,...,-1.86,4.24,6.10,1.86,3.98,2.12,0.47,3.73,0.00,2.59
1,2,-1.30,-0.43,304.80,152.40,1.97,0.00,7.39,0.98,3.69,...,-2.11,4.71,6.83,2.11,4.47,2.36,0.42,4.24,2.52,2.59
2,3,-1.30,-0.08,171.40,85.71,1.59,1.39,7.56,0.80,3.78,...,-1.75,4.42,6.17,1.75,3.96,2.21,0.45,3.56,0.01,2.57
3,4,-1.30,-0.61,393.90,197.00,1.86,0.00,6.59,0.93,3.30,...,-2.50,5.10,7.60,2.50,5.05,2.55,0.39,5.00,0.00,2.58
4,5,-1.30,-0.27,260.50,130.30,1.49,0.00,6.76,0.74,3.38,...,-2.03,4.93,6.97,2.03,4.50,2.47,0.41,4.11,2.25,2.58


,file,exists,size_bytes,absolute_path
0,All_molecules_noncorelated_descriptors_and_y_q...,True,624064,C:\Users\plato\PycharmProjects\DNA\All_molecul...
1,62_descriptors_and_quantum.txt,True,1377,C:\Users\plato\PycharmProjects\DNA\62_descript...
2,test_molecules.csv,True,85,C:\Users\plato\PycharmProjects\DNA\test_molecu...


## 2. Descriptor list loading and mapping

Descriptor names are read from `62_descriptors.txt`. Empty lines and lines beginning with `#` are ignored, and duplicate entries are removed while preserving their original order.

Descriptor names are matched to the Excel columns using exact matching, whitespace-normalized matching, and case-insensitive matching. The resulting tables document successfully matched descriptors and descriptors absent from the input dataset.

`normalize_descriptor_name(x)` standardizes whitespace before descriptor matching.


In [3]:
# ============================================================
# Descriptor list loading and column mapping
# ============================================================

with open(DESCRIPTOR_FILE, "r", encoding="utf-8") as f:
    descriptor_lines = [line.strip() for line in f.readlines()]

selected_descriptors_txt = []
for line in descriptor_lines:
    if line and not line.startswith("#"):
        selected_descriptors_txt.append(line)
selected_descriptors_txt = list(dict.fromkeys(selected_descriptors_txt))

if len(selected_descriptors_txt) == 0:
    raise ValueError("62_descriptors.txt was found but contains no descriptor names.")

excel_columns = list(raw_df.columns)


def normalize_descriptor_name(x):
    return re.sub(r"\s+", " ", str(x).strip())


exact_column_map = {str(c): c for c in excel_columns}
stripped_column_map = {normalize_descriptor_name(c): c for c in excel_columns}
lower_column_map = {normalize_descriptor_name(c).lower(): c for c in excel_columns}

mapping_records = []
found_descriptors = []
missing_descriptors = []

for desc in selected_descriptors_txt:
    desc_norm = normalize_descriptor_name(desc)
    excel_col = None
    match_type = "missing"

    if desc in exact_column_map:
        excel_col = exact_column_map[desc]
        match_type = "exact"
    elif desc_norm in stripped_column_map:
        excel_col = stripped_column_map[desc_norm]
        match_type = "trimmed"
    elif desc_norm.lower() in lower_column_map:
        excel_col = lower_column_map[desc_norm.lower()]
        match_type = "case_insensitive"

    if excel_col is not None:
        found_descriptors.append(excel_col)
    else:
        missing_descriptors.append(desc)

    mapping_records.append({
        "descriptor_from_txt": desc,
        "excel_column": excel_col if excel_col is not None else "not found",
        "match_type": match_type
    })

found_descriptors = list(dict.fromkeys(found_descriptors))
TABLES["descriptor_mapping"] = pd.DataFrame(mapping_records)
TABLES["missing_descriptors"] = pd.DataFrame({"missing_descriptor": missing_descriptors})

if len(found_descriptors) == 0:
    raise ValueError("None of the descriptors from 62_descriptors.txt were found in the Excel file.")

save_dataframe(TABLES["descriptor_mapping"], "descriptor_mapping.xlsx")
save_dataframe(TABLES["missing_descriptors"], "missing_descriptors.xlsx")

print(f"Descriptors in TXT: {len(selected_descriptors_txt)}")
print(f"Descriptors found in Excel: {len(found_descriptors)}")
print(f"Missing descriptors: {len(missing_descriptors)}")
display(TABLES["descriptor_mapping"].head(30))
display(TABLES["missing_descriptors"])


Descriptors in TXT: 73
Descriptors found in Excel: 73
Missing descriptors: 0


,descriptor_from_txt,excel_column,match_type
0,ASP:(alvaDesc),ASP:(alvaDesc),exact
1,ChiA_X:(alvaDesc),ChiA_X:(alvaDesc),exact
2,DISPm:(alvaDesc),DISPm:(alvaDesc),exact
3,DISPp:(alvaDesc),DISPp:(alvaDesc),exact
4,DLS_cons:(alvaDesc),DLS_cons:(alvaDesc),exact
5,Dp:(alvaDesc),Dp:(alvaDesc),exact
6,Ds:(alvaDesc),Ds:(alvaDesc),exact
7,E2u:(alvaDesc),E2u:(alvaDesc),exact
8,F01[C-C]:(alvaDesc),F01[C-C]:(alvaDesc),exact
9,F03[C-X]:(alvaDesc),F03[C-X]:(alvaDesc),exact


,missing_descriptor


## 3. Target definition, molecule annotation, and fixed external split

The target column and molecular identifiers are identified from the input table. If no suitable identifier column is present, sequential `Molecule_ID` values are assigned.

Molecule classes are assigned manually according to the dataset structure:

- `Molecule_ID` 1–6: bimetallic clusters;
- `Molecule_ID` 7, 26, 45, and 73: individual bases;
- all remaining molecules: bimetal–base complexes.

The predefined `EXTERNAL_TEST_IDS` are used to construct the fixed external test set. If `test_molecules.csv` is available, its molecule IDs are checked against the configured test set.

`assign_molecule_type_and_name(molecule_id)` provides the molecule class and structure label used in the complex-only validation scheme.


In [4]:
# ============================================================
# Target definition, molecule annotation, and fixed external split
# ============================================================

target_col = None
for candidate in TARGET_COLUMN_CANDIDATES:
    if candidate in raw_df.columns:
        target_col = candidate
        break

if target_col is None:
    lower_to_original = {str(c).lower(): c for c in raw_df.columns}
    for candidate in TARGET_COLUMN_CANDIDATES:
        if candidate.lower() in lower_to_original:
            target_col = lower_to_original[candidate.lower()]
            break

if target_col is None:
    raise ValueError(f"Target column was not found. Expected one of: {TARGET_COLUMN_CANDIDATES}")

possible_id_columns = [
    "Molecule_ID", "Molecule ID", "Mol_ID", "MolID", "ID", "No", "Number",
    "molecule_id", "mol_id", "index"
]

molecule_id_col = None
for candidate in possible_id_columns:
    if candidate in raw_df.columns:
        values = pd.to_numeric(raw_df[candidate], errors="coerce")
        if values.notna().sum() == len(raw_df) and values.nunique() == len(raw_df):
            molecule_id_col = candidate
            break

df = raw_df.copy()
if molecule_id_col is None:
    df.insert(0, "Molecule_ID", np.arange(1, len(df) + 1))
    molecule_id_col = "Molecule_ID_created_from_row_number"
else:
    if molecule_id_col != "Molecule_ID":
        df.insert(0, "Molecule_ID", pd.to_numeric(df[molecule_id_col], errors="coerce").astype(int))
    else:
        df["Molecule_ID"] = pd.to_numeric(df["Molecule_ID"], errors="coerce").astype(int)

df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
if df[target_col].isna().any():
    bad_ids = df.loc[df[target_col].isna(), "Molecule_ID"].tolist()
    raise ValueError(f"Target column {target_col} contains NaN/non-numeric values for Molecule_ID: {bad_ids}")

all_ids = set(df["Molecule_ID"].astype(int).tolist())
missing_test_ids = [x for x in EXTERNAL_TEST_IDS if x not in all_ids]
if missing_test_ids:
    raise ValueError(f"External test Molecule_ID values were not found in data: {missing_test_ids}")

if TEST_FILE.exists():
    test_df_raw_from_file = pd.read_csv(TEST_FILE)
    numeric_values = []
    for col in test_df_raw_from_file.columns:
        numeric_values.extend(pd.to_numeric(test_df_raw_from_file[col], errors="coerce").dropna().astype(int).tolist())
    numeric_values = list(dict.fromkeys(numeric_values))
    file_set = set(numeric_values)
    hard_set = set(EXTERNAL_TEST_IDS)
    if file_set != hard_set:
        add_warning(
            "test_molecules.csv does not exactly match the hard-coded external test IDs. Hard-coded IDs are used.")
    test_file_audit = pd.DataFrame([{
        "test_file_found": True,
        "ids_from_test_file": numeric_values,
        "hardcoded_external_test_ids": EXTERNAL_TEST_IDS,
        "matches_hardcoded": file_set == hard_set,
        "ids_in_file_not_in_hardcoded": sorted(list(file_set - hard_set)),
        "ids_in_hardcoded_not_in_file": sorted(list(hard_set - file_set))
    }])
else:
    add_warning("test_molecules.csv was not found. Hard-coded external test IDs are used.")
    test_file_audit = pd.DataFrame([{
        "test_file_found": False,
        "ids_from_test_file": [],
        "hardcoded_external_test_ids": EXTERNAL_TEST_IDS,
        "matches_hardcoded": False,
        "ids_in_file_not_in_hardcoded": [],
        "ids_in_hardcoded_not_in_file": EXTERNAL_TEST_IDS
    }])

TABLES["raw_file_audit"] = pd.concat([TABLES["raw_file_audit"], test_file_audit], axis=0, ignore_index=True)


def assign_molecule_type_and_name(molecule_id: int):
    molecule_id = int(molecule_id)
    if 1 <= molecule_id <= 6:
        return "bimetal", f"bimetal_{molecule_id}"
    base_names = {
        7: "cytosine",
        26: "tyrosine_or_thymine",
        45: "adenine",
        73: "guanine"
    }
    if molecule_id in base_names:
        return "base", base_names[molecule_id]
    return "complex", f"complex_{molecule_id}"


assigned = df["Molecule_ID"].apply(assign_molecule_type_and_name)
df["Molecule_Type"] = [x[0] for x in assigned]
df["Structure_Name"] = [x[1] for x in assigned]
TABLES["molecule_type_assignment"] = df[["Molecule_ID", "Molecule_Type", "Structure_Name"]].copy()

df["Is_External_Test"] = df["Molecule_ID"].isin(EXTERNAL_TEST_IDS)
train_df_raw = df.loc[~df["Is_External_Test"]].copy()
test_df_raw = df.loc[df["Is_External_Test"]].copy()

if len(test_df_raw) != len(EXTERNAL_TEST_IDS):
    raise ValueError(f"External test size mismatch. Expected {len(EXTERNAL_TEST_IDS)}, got {len(test_df_raw)}.")

split_table = df[["Molecule_ID", "Molecule_Type", "Structure_Name", "Is_External_Test"]].copy()
split_table["Split"] = np.where(split_table["Is_External_Test"], "external_test", "train_internal")
TABLES["train_test_split"] = split_table

train_ids = train_df_raw["Molecule_ID"].astype(int).tolist()
test_ids = test_df_raw["Molecule_ID"].astype(int).tolist()
with open(versioned_path(RESULTS_DIR / "train_test_ids.json", OVERWRITE_EXISTING_RESULTS), "w", encoding="utf-8") as f:
    json.dump({"train_ids": train_ids, "external_test_ids": test_ids}, f, ensure_ascii=False, indent=2)

print(f"Target column: {target_col}")
print(f"Molecule_ID source: {molecule_id_col}")
print(f"Train/internal size: {len(train_df_raw)}")
print(f"External test size: {len(test_df_raw)}")
display(pd.crosstab(split_table["Split"], split_table["Molecule_Type"]))


Target column: Log_S1
Molecule_ID source: Molecule_ID_created_from_row_number
Train/internal size: 80
External test size: 20


Molecule_Type,base,bimetal,complex
Split,,,
external_test,0,0,20
train_internal,4,6,70


## 4. Train-only preprocessing

All preprocessing rules are estimated from the training data and then applied unchanged to the external test set.

The procedure includes conversion to numeric values, replacement of infinite values with `NaN`, removal of descriptors with more than 50% missing values in the training set, median imputation using training-set medians, and removal of descriptors that are constant in the training data.

The resulting matrices are stored as `X_train_pre` and `X_test_pre`. A preprocessing summary and the imputation medians are saved for reproducibility.


In [5]:
# ============================================================
# Train-only preprocessing
# ============================================================

X_train_raw = train_df_raw[found_descriptors].copy().replace([np.inf, -np.inf], np.nan)
X_test_raw = test_df_raw[found_descriptors].copy().replace([np.inf, -np.inf], np.nan)
y_train = train_df_raw[target_col].astype(float).reset_index(drop=True)
y_test = test_df_raw[target_col].astype(float).reset_index(drop=True)

for col in X_train_raw.columns:
    X_train_raw[col] = pd.to_numeric(X_train_raw[col], errors="coerce")
    X_test_raw[col] = pd.to_numeric(X_test_raw[col], errors="coerce")

preprocessing_records = []
nan_fraction_train = X_train_raw.isna().mean()
features_high_nan = nan_fraction_train[nan_fraction_train > 0.50].index.tolist()
preprocessing_records.append({
    "step": "remove_high_nan_features",
    "criterion": "NaN fraction on train > 0.50",
    "n_features_before": X_train_raw.shape[1],
    "n_features_removed": len(features_high_nan),
    "removed_features": "; ".join(features_high_nan)
})

X_train_clean = X_train_raw.drop(columns=features_high_nan)
X_test_clean = X_test_raw.drop(columns=features_high_nan)
train_medians = X_train_clean.median(axis=0, skipna=True)

all_nan_features = train_medians[train_medians.isna()].index.tolist()
if all_nan_features:
    preprocessing_records.append({
        "step": "remove_all_nan_features",
        "criterion": "median is NaN on train",
        "n_features_before": X_train_clean.shape[1],
        "n_features_removed": len(all_nan_features),
        "removed_features": "; ".join(all_nan_features)
    })
    X_train_clean = X_train_clean.drop(columns=all_nan_features)
    X_test_clean = X_test_clean.drop(columns=all_nan_features)
    train_medians = train_medians.drop(index=all_nan_features)

X_train_imputed = X_train_clean.fillna(train_medians)
X_test_imputed = X_test_clean.fillna(train_medians)
constant_features = [col for col in X_train_imputed.columns if X_train_imputed[col].nunique(dropna=False) <= 1]
preprocessing_records.append({
    "step": "remove_constant_features",
    "criterion": "nunique on train <= 1",
    "n_features_before": X_train_imputed.shape[1],
    "n_features_removed": len(constant_features),
    "removed_features": "; ".join(constant_features)
})

X_train_pre = X_train_imputed.drop(columns=constant_features).reset_index(drop=True)
X_test_pre = X_test_imputed.drop(columns=constant_features).reset_index(drop=True)

if X_train_pre.shape[1] == 0:
    raise ValueError("No descriptors remain after preprocessing.")

train_meta = train_df_raw[["Molecule_ID", "Molecule_Type", "Structure_Name"]].reset_index(drop=True).copy()
test_meta = test_df_raw[["Molecule_ID", "Molecule_Type", "Structure_Name"]].reset_index(drop=True).copy()

TABLES["preprocessing_report"] = pd.DataFrame(preprocessing_records)
save_dataframe(TABLES["preprocessing_report"], "preprocessing_report.xlsx")
save_dataframe(pd.DataFrame({"feature": train_medians.index, "train_median": train_medians.values}),
               "train_medians_used_for_imputation.xlsx")

print(f"Initial found descriptors: {len(found_descriptors)}")
print(f"Descriptors after basic preprocessing: {X_train_pre.shape[1]}")
display(TABLES["preprocessing_report"])


Initial found descriptors: 73
Descriptors after basic preprocessing: 73


,step,criterion,n_features_before,n_features_removed,removed_features
0,remove_high_nan_features,NaN fraction on train > 0.50,73,0,
1,remove_constant_features,nunique on train <= 1,73,0,


## 5. Correlation, VIF, and target-distribution diagnostics

Pairwise Pearson correlations and variance inflation factors (VIF) are calculated using the training set only.

`compute_correlation_pairs(X)` summarizes all pairwise correlations, while `iterative_correlation_filter(X, cutoff)` optionally removes descriptors involved in correlations above the specified threshold. VIF values are calculated after standardization, and `iterative_vif_filter(X, cutoff)` can be used for iterative VIF-based reduction.

Correlation and VIF filtering are controlled by `DO_CORRELATION_FILTERING` and `DO_VIF_FILTERING`. Diagnostic tables and figures are retained regardless of whether filtering is enabled.

The training-set distribution of `Log_S1` is also summarized by its range, mean, standard deviation, and skewness.


In [6]:
# ============================================================
# Correlation and VIF diagnostics
# ============================================================


def compute_correlation_pairs(X: pd.DataFrame) -> pd.DataFrame:
    corr = X.corr(method="pearson")
    records = []
    cols = corr.columns.tolist()
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            records.append({
                "feature_1": cols[i],
                "feature_2": cols[j],
                "pearson_r": corr.iloc[i, j],
                "abs_pearson_r": abs(corr.iloc[i, j])
            })
    return pd.DataFrame(records).sort_values("abs_pearson_r", ascending=False)


def iterative_correlation_filter(X: pd.DataFrame, cutoff: float):
    X_current = X.copy()
    removed = []
    while X_current.shape[1] > 1:
        corr = X_current.corr().abs()
        np.fill_diagonal(corr.values, 0)
        max_corr = corr.max().max()
        if pd.isna(max_corr) or max_corr <= cutoff:
            break
        pair = np.where(corr.values == max_corr)
        i, j = int(pair[0][0]), int(pair[1][0])
        f1, f2 = corr.index[i], corr.columns[j]
        remove_feature = f1 if corr[f1].mean() >= corr[f2].mean() else f2
        removed.append({
            "removed_feature": remove_feature,
            "paired_feature_1": f1,
            "paired_feature_2": f2,
            "pair_abs_r": max_corr,
            "reason": "higher mean absolute correlation"
        })
        X_current = X_current.drop(columns=[remove_feature])
    return X_current.columns.tolist(), pd.DataFrame(removed)


correlation_pairs = compute_correlation_pairs(X_train_pre)
high_corr_pairs = correlation_pairs[correlation_pairs["abs_pearson_r"] > CORRELATION_CUTOFF].copy()
TABLES["correlation_pairs"] = correlation_pairs
TABLES["high_correlation_pairs"] = high_corr_pairs
save_dataframe(correlation_pairs, "correlation_pairs.xlsx")
save_dataframe(high_corr_pairs, "high_correlation_pairs.xlsx")

fig, ax = plt.subplots(figsize=(max(8, 0.25 * X_train_pre.shape[1]), max(6, 0.25 * X_train_pre.shape[1])))
sns.heatmap(X_train_pre.corr(), cmap="vlag", center=0, square=False, cbar_kws={"label": "Pearson r"}, ax=ax)
ax.set_title("Descriptor correlation heatmap, train only")
save_figure(fig, "correlation_heatmap_train")

features_after_corr = list(X_train_pre.columns)
if DO_CORRELATION_FILTERING:
    features_after_corr, corr_removed = iterative_correlation_filter(X_train_pre, CORRELATION_CUTOFF)
    save_dataframe(corr_removed, "correlation_filter_removed_features.xlsx")
    add_warning(
        f"Correlation filtering is ON. Removed {len(corr_removed)} features using train-only cutoff {CORRELATION_CUTOFF}.")
else:
    print("Correlation filtering is OFF. Correlation is diagnostic only.")

X_train_corr = X_train_pre[features_after_corr].copy()
X_test_corr = X_test_pre[features_after_corr].copy()


def calculate_vif_table(X: pd.DataFrame) -> pd.DataFrame:
    if not HAS_STATSMODELS:
        add_warning("statsmodels is unavailable; VIF cannot be calculated.")
        return pd.DataFrame(columns=["feature", "VIF"])
    X_num = X.loc[:, X.std(axis=0) > 0].copy()
    if X_num.shape[1] == 0:
        return pd.DataFrame(columns=["feature", "VIF"])
    X_scaled = pd.DataFrame(StandardScaler().fit_transform(X_num), columns=X_num.columns, index=X_num.index)
    records = []
    for i, col in enumerate(X_scaled.columns):
        try:
            vif_value = variance_inflation_factor(X_scaled.values, i)
        except Exception:
            vif_value = np.inf
        records.append({"feature": col, "VIF": vif_value})
    return pd.DataFrame(records).sort_values("VIF", ascending=False)


def iterative_vif_filter(X: pd.DataFrame, cutoff: float):
    X_current = X.copy()
    removed = []
    while X_current.shape[1] > 2:
        vif_df = calculate_vif_table(X_current)
        if vif_df.empty:
            break
        first = vif_df.iloc[0]
        max_vif = first["VIF"]
        if not np.isinf(max_vif) and (pd.isna(max_vif) or max_vif <= cutoff):
            break
        feature_to_remove = first["feature"]
        removed.append({"removed_feature": feature_to_remove, "VIF": max_vif, "criterion": f"VIF > {cutoff}"})
        X_current = X_current.drop(columns=[feature_to_remove])
    return X_current.columns.tolist(), pd.DataFrame(removed)


vif_before = calculate_vif_table(X_train_corr)
high_vif = vif_before[vif_before["VIF"] > VIF_CUTOFF].copy() if not vif_before.empty else pd.DataFrame(
    columns=["feature", "VIF"])
TABLES["vif_before"] = vif_before
TABLES["high_vif"] = high_vif
save_dataframe(vif_before, "vif_before.xlsx")
save_dataframe(high_vif, "high_vif.xlsx")

if not vif_before.empty:
    vif_plot_df = vif_before.copy()
    vif_plot_df["VIF_plot"] = vif_plot_df["VIF"].replace(np.inf, np.nan)
    vif_plot_df = vif_plot_df.sort_values("VIF_plot", ascending=True).tail(40)
    fig, ax = plt.subplots(figsize=(9, max(5, 0.25 * len(vif_plot_df))))
    ax.barh(vif_plot_df["feature"], vif_plot_df["VIF_plot"])
    ax.axvline(VIF_CUTOFF, linestyle="--", label=f"VIF cutoff = {VIF_CUTOFF}")
    ax.set_xlabel("VIF")
    ax.set_title("Top VIF values, train only")
    ax.legend()
    save_figure(fig, "vif_barplot_train")

features_after_vif = list(X_train_corr.columns)
if DO_VIF_FILTERING:
    features_after_vif, vif_removed = iterative_vif_filter(X_train_corr, VIF_CUTOFF)
    save_dataframe(vif_removed, "vif_filter_removed_features.xlsx")
    add_warning(f"VIF filtering is ON. Removed {len(vif_removed)} features using train-only cutoff {VIF_CUTOFF}.")
else:
    print("VIF filtering is OFF. VIF is diagnostic only.")

X_train_final_base = X_train_corr[features_after_vif].reset_index(drop=True).copy()
X_test_final_base = X_test_corr[features_after_vif].reset_index(drop=True).copy()

# Target distribution summary
target_stats = {
    "target_column": target_col,
    "n_train": len(y_train),
    "min": float(y_train.min()),
    "max": float(y_train.max()),
    "mean": float(y_train.mean()),
    "std": float(y_train.std(ddof=1)),
    "skewness": float(stats.skew(y_train, nan_policy="omit"))
}
target_stats_df = pd.DataFrame([target_stats])
save_dataframe(target_stats_df, "target_distribution_stats.xlsx")
if abs(target_stats["skewness"]) > 1.0:
    add_warning(
        f"Train target skewness is {target_stats['skewness']:.3f}. Endpoint is already Log_S1; no additional transform is applied by default.")

fig, ax = plt.subplots(figsize=(7, 5))
sns.histplot(y_train, kde=True, ax=ax)
ax.set_xlabel(target_col)
ax.set_ylabel("Count")
ax.set_title("Train target distribution")
save_figure(fig, "target_distribution_train")

print(f"High-correlation pairs |r| > {CORRELATION_CUTOFF}: {len(high_corr_pairs)}")
print(f"High-VIF features VIF > {VIF_CUTOFF}: {len(high_vif)}")
display(target_stats_df)


High-correlation pairs |r| > 0.95: 5
High-VIF features VIF > 10.0: 69


,target_column,n_train,min,max,mean,std,skewness
0,Log_S1,80,2.36,2.88,2.58,0.08,0.50


## 6. QSPR metrics and complex-only cross-validation

Model performance is evaluated using a repeated cross-validation scheme designed for the composition of this dataset. Validation folds contain only bimetal–base complexes, whereas isolated bimetallic clusters and individual bases remain in the training portion of every split. The external test set is not used during internal validation.

The workflow reports Q², RMSE, and MAE for repeated cross-validation, together with training and external-test metrics. External validation additionally includes the concordance correlation coefficient, Q²F1–Q²F3, and Golbraikh–Tropsha criteria.

`make_complex_only_cv_splits` generates the repeated validation partitions, `evaluate_cv_model` evaluates internal predictive performance, and `evaluate_train_test` calculates final train and external-test metrics.


In [7]:
# ============================================================
# Performance metrics and complex-only cross-validation
# ============================================================


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def adjusted_r2_score(y_true, y_pred, p):
    n = len(y_true)
    if n <= p + 1:
        return np.nan
    r2 = r2_score(y_true, y_pred)
    return float(1 - (1 - r2) * (n - 1) / (n - p - 1))


def concordance_correlation_coefficient(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true, ddof=1)
    var_pred = np.var(y_pred, ddof=1)
    covariance = np.cov(y_true, y_pred, ddof=1)[0, 1]
    denominator = var_true + var_pred + (mean_true - mean_pred) ** 2
    return np.nan if denominator == 0 else float((2 * covariance) / denominator)


def q2_f1(y_train_true, y_test_true, y_test_pred):
    denominator = np.sum((y_test_true - np.mean(y_train_true)) ** 2)
    return np.nan if denominator == 0 else float(1 - np.sum((y_test_true - y_test_pred) ** 2) / denominator)


def q2_f2(y_test_true, y_test_pred):
    denominator = np.sum((y_test_true - np.mean(y_test_true)) ** 2)
    return np.nan if denominator == 0 else float(1 - np.sum((y_test_true - y_test_pred) ** 2) / denominator)


def q2_f3(y_train_true, y_test_true, y_test_pred):
    train_variance_term = np.sum((y_train_true - np.mean(y_train_true)) ** 2) / max(len(y_train_true), 1)
    mse_test = np.sum((y_test_true - y_test_pred) ** 2) / max(len(y_test_true), 1)
    return np.nan if train_variance_term == 0 else float(1 - mse_test / train_variance_term)


def golbraikh_tropsha_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2:
        return {
            "GT_r2": np.nan, "GT_r0_squared": np.nan, "GT_r0prime_squared": np.nan,
            "GT_k": np.nan, "GT_kprime": np.nan,
            "GT_abs_r2_minus_r0sq_over_r2": np.nan,
            "GT_abs_r2_minus_r0psq_over_r2": np.nan,
            "GT_criteria_pass": False
        }
    r2 = r2_score(y_true, y_pred)
    denom_k = np.sum(y_pred ** 2)
    k = np.sum(y_true * y_pred) / denom_k if denom_k != 0 else np.nan
    denom_kp = np.sum(y_true ** 2)
    kprime = np.sum(y_true * y_pred) / denom_kp if denom_kp != 0 else np.nan
    y_pred_scaled = k * y_pred if not pd.isna(k) else np.full_like(y_pred, np.nan)
    y_true_scaled = kprime * y_true if not pd.isna(kprime) else np.full_like(y_true, np.nan)
    denom_true = np.sum((y_true - np.mean(y_true)) ** 2)
    denom_pred = np.sum((y_pred - np.mean(y_pred)) ** 2)
    r0_squared = 1 - np.sum((y_true - y_pred_scaled) ** 2) / denom_true if denom_true != 0 else np.nan
    r0prime_squared = 1 - np.sum((y_pred - y_true_scaled) ** 2) / denom_pred if denom_pred != 0 else np.nan
    ratio1 = np.nan if r2 == 0 or pd.isna(r2) else abs(r2 - r0_squared) / abs(r2)
    ratio2 = np.nan if r2 == 0 or pd.isna(r2) else abs(r2 - r0prime_squared) / abs(r2)
    criteria_pass = bool(
        (r2 > 0.6 if not pd.isna(r2) else False) and
        ((ratio1 < 0.1 if not pd.isna(ratio1) else False) or (ratio2 < 0.1 if not pd.isna(ratio2) else False)) and
        (0.85 <= k <= 1.15 if not pd.isna(k) else False) and
        (0.85 <= kprime <= 1.15 if not pd.isna(kprime) else False)
    )
    return {
        "GT_r2": float(r2),
        "GT_r0_squared": float(r0_squared) if not pd.isna(r0_squared) else np.nan,
        "GT_r0prime_squared": float(r0prime_squared) if not pd.isna(r0prime_squared) else np.nan,
        "GT_k": float(k) if not pd.isna(k) else np.nan,
        "GT_kprime": float(kprime) if not pd.isna(kprime) else np.nan,
        "GT_abs_r2_minus_r0sq_over_r2": float(ratio1) if not pd.isna(ratio1) else np.nan,
        "GT_abs_r2_minus_r0psq_over_r2": float(ratio2) if not pd.isna(ratio2) else np.nan,
        "GT_criteria_pass": criteria_pass
    }


def predict_1d(model, X):
    pred = model.predict(X)
    return np.asarray(pred).reshape(-1)


def make_complex_only_cv_splits(meta: pd.DataFrame, n_splits=5, n_repeats=20, random_state=42):
    meta_reset = meta.reset_index(drop=True)
    complex_positions = np.where(meta_reset["Molecule_Type"].values == "complex")[0]
    fixed_train_positions = np.where(meta_reset["Molecule_Type"].values != "complex")[0]
    if len(complex_positions) < 2:
        raise ValueError("Not enough complex molecules for complex-only CV.")
    actual_splits = min(n_splits, len(complex_positions))
    splits = []
    for repeat in range(n_repeats):
        kf = KFold(n_splits=actual_splits, shuffle=True, random_state=random_state + repeat)
        for complex_train_local, complex_val_local in kf.split(complex_positions):
            train_complex_positions = complex_positions[complex_train_local]
            val_positions = complex_positions[complex_val_local]
            train_positions = np.concatenate([fixed_train_positions, train_complex_positions])
            splits.append((np.sort(train_positions), np.sort(val_positions)))
    return splits


def evaluate_cv_model(estimator, X, y, meta, n_splits=5, n_repeats=20, random_state=42):
    splits = make_complex_only_cv_splits(meta, n_splits, n_repeats, random_state)
    fold_records = []
    for fold_id, (tr_idx, val_idx) in enumerate(splits, start=1):
        model = clone(estimator)
        try:
            model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
            pred_val = predict_1d(model, X.iloc[val_idx])
            fold_records.append({
                "fold_id": fold_id,
                "Q2": r2_score(y.iloc[val_idx], pred_val) if len(val_idx) > 1 else np.nan,
                "RMSE": rmse(y.iloc[val_idx], pred_val),
                "MAE": mean_absolute_error(y.iloc[val_idx], pred_val),
                "n_train_fold": len(tr_idx),
                "n_val_fold": len(val_idx)
            })
        except Exception as e:
            fold_records.append({
                "fold_id": fold_id, "Q2": np.nan, "RMSE": np.nan, "MAE": np.nan,
                "n_train_fold": len(tr_idx), "n_val_fold": len(val_idx), "error": str(e)
            })
    fold_df = pd.DataFrame(fold_records)
    summary = {
        "Q2_mean": fold_df["Q2"].mean(skipna=True),
        "Q2_std": fold_df["Q2"].std(skipna=True),
        "RMSE_CV_mean": fold_df["RMSE"].mean(skipna=True),
        "RMSE_CV_std": fold_df["RMSE"].std(skipna=True),
        "MAE_CV_mean": fold_df["MAE"].mean(skipna=True),
        "MAE_CV_std": fold_df["MAE"].std(skipna=True),
        "n_cv_folds": len(fold_df),
        "n_failed_folds": int(fold_df[["Q2", "RMSE", "MAE"]].isna().any(axis=1).sum())
    }
    return summary, fold_df


def evaluate_train_test(estimator, X_train, y_train_local, X_test, y_test_local, p):
    model = clone(estimator)
    model.fit(X_train, y_train_local)
    pred_train = predict_1d(model, X_train)
    pred_test = predict_1d(model, X_test)
    metrics = {
        "R2_train": r2_score(y_train_local, pred_train),
        "adjusted_R2_train": adjusted_r2_score(y_train_local, pred_train, p),
        "RMSE_train": rmse(y_train_local, pred_train),
        "MAE_train": mean_absolute_error(y_train_local, pred_train),
        "R2_test": r2_score(y_test_local, pred_test) if len(y_test_local) > 1 else np.nan,
        "RMSE_test": rmse(y_test_local, pred_test),
        "MAE_test": mean_absolute_error(y_test_local, pred_test),
        "CCC": concordance_correlation_coefficient(y_test_local, pred_test),
        "Q2F1": q2_f1(y_train_local, y_test_local, pred_test),
        "Q2F2": q2_f2(y_test_local, pred_test),
        "Q2F3": q2_f3(y_train_local, y_test_local, pred_test)
    }
    metrics.update(golbraikh_tropsha_metrics(y_test_local, pred_test))
    pred_train_df = pd.DataFrame({
        "Molecule_ID": train_meta["Molecule_ID"].values,
        "Molecule_Type": train_meta["Molecule_Type"].values,
        "Structure_Name": train_meta["Structure_Name"].values,
        "Split": "train",
        "Experimental": y_train_local.values,
        "Predicted": pred_train,
        "Residual": y_train_local.values - pred_train
    })
    pred_test_df = pd.DataFrame({
        "Molecule_ID": test_meta["Molecule_ID"].values,
        "Molecule_Type": test_meta["Molecule_Type"].values,
        "Structure_Name": test_meta["Structure_Name"].values,
        "Split": "external_test",
        "Experimental": y_test_local.values,
        "Predicted": pred_test,
        "Residual": y_test_local.values - pred_test
    })
    predictions = pd.concat([pred_train_df, pred_test_df], axis=0, ignore_index=True)
    return model, metrics, predictions


print("Metrics and complex-only CV helpers are ready.")


Metrics and complex-only CV helpers are ready.


## 7. Model definitions and Optuna search spaces

This section defines the regression algorithms and their hyperparameter spaces. Models that require feature scaling are wrapped with `StandardScaler`; tree-based models are fitted without scaling.

The model set includes MLR, PLS, Ridge, LASSO, ElasticNet, SVR, Random Forest, Extra Trees, Gradient Boosting, XGBoost, LightGBM, CatBoost, Gaussian Process regression, KNN, and AdaBoost.

Optional libraries are checked before model construction. Models whose dependencies are unavailable are excluded from `AVAILABLE_MODEL_NAMES`.

`build_estimator(model_name, trial, n_features, n_train_samples)` returns either an estimator with default settings or an Optuna-configured estimator when a trial object is supplied.


In [8]:
# ============================================================
# Model definitions and hyperparameter spaces
# ============================================================


def model_is_available(model_name: str) -> bool:
    if model_name == "XGBoost":
        return HAS_XGBOOST
    if model_name == "LightGBM":
        return HAS_LIGHTGBM
    if model_name == "CatBoost":
        return HAS_CATBOOST
    return True


def wrap_with_scaler_if_needed(model_name: str, model):
    models_requiring_scaling = {"MLR", "PLS", "Ridge", "LASSO", "ElasticNet", "SVR", "GaussianProcess", "KNN"}
    if model_name in models_requiring_scaling:
        return Pipeline([("scaler", StandardScaler()), ("model", model)])
    return model


def build_adaboost(base, n_estimators, learning_rate):
    try:
        return AdaBoostRegressor(estimator=base, n_estimators=n_estimators, learning_rate=learning_rate,
                                 random_state=RANDOM_STATE)
    except TypeError:
        return AdaBoostRegressor(base_estimator=base, n_estimators=n_estimators, learning_rate=learning_rate,
                                 random_state=RANDOM_STATE)


def build_estimator(model_name: str, trial=None, n_features: int = None, n_train_samples: int = None):
    if n_features is None:
        n_features = X_train_final_base.shape[1]
    if n_train_samples is None:
        n_train_samples = len(y_train)

    if model_name == "MLR":
        model = LinearRegression()

    elif model_name == "PLS":
        max_comp = max(1, min(n_features, n_train_samples - 2, 15))
        n_comp = trial.suggest_int("n_components", 1, max_comp) if trial is not None else min(2, max_comp)
        model = PLSRegression(n_components=n_comp)

    elif model_name == "Ridge":
        alpha = trial.suggest_float("alpha", 1e-5, 1e3, log=True) if trial is not None else 1.0
        model = Ridge(alpha=alpha, random_state=RANDOM_STATE)

    elif model_name == "LASSO":
        alpha = trial.suggest_float("alpha", 1e-5, 10.0, log=True) if trial is not None else 0.01
        model = Lasso(alpha=alpha, max_iter=200000, random_state=RANDOM_STATE)

    elif model_name == "ElasticNet":
        alpha = trial.suggest_float("alpha", 1e-5, 10.0, log=True) if trial is not None else 0.01
        l1_ratio = trial.suggest_float("l1_ratio", 0.05, 0.95) if trial is not None else 0.5
        model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=200000, random_state=RANDOM_STATE)

    elif model_name == "SVR":
        if trial is not None:
            C = trial.suggest_float("C", 1e-2, 1e3, log=True)
            epsilon = trial.suggest_float("epsilon", 1e-4, 1.0, log=True)
            kernel = trial.suggest_categorical("kernel", ["rbf", "linear", "poly"])
            gamma = trial.suggest_float("gamma", 1e-4, 10.0, log=True) if kernel in ["rbf", "poly"] else "scale"
        else:
            C, epsilon, kernel, gamma = 10.0, 0.1, "rbf", "scale"
        model = SVR(C=C, epsilon=epsilon, kernel=kernel, gamma=gamma)

    elif model_name == "RandomForest":
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 150, 900) if trial is not None else 500,
            "max_depth": trial.suggest_categorical("max_depth",
                                                   [None, 2, 3, 4, 5, 6, 8, 10]) if trial is not None else None,
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8) if trial is not None else 1,
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 12) if trial is not None else 2,
            "max_features": trial.suggest_float("max_features", 0.3, 1.0) if trial is not None else 1.0,
        }
        model = RandomForestRegressor(**params, random_state=RANDOM_STATE, n_jobs=-1)

    elif model_name == "ExtraTrees":
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 150, 1000) if trial is not None else 600,
            "max_depth": trial.suggest_categorical("max_depth",
                                                   [None, 2, 3, 4, 5, 6, 8, 10]) if trial is not None else None,
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8) if trial is not None else 1,
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 12) if trial is not None else 2,
            "max_features": trial.suggest_float("max_features", 0.3, 1.0) if trial is not None else 1.0,
        }
        model = ExtraTreesRegressor(**params, random_state=RANDOM_STATE, n_jobs=-1)

    elif model_name == "GradientBoosting":
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 80, 700) if trial is not None else 300,
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True) if trial is not None else 0.03,
            "max_depth": trial.suggest_int("max_depth", 1, 4) if trial is not None else 2,
            "subsample": trial.suggest_float("subsample", 0.5, 1.0) if trial is not None else 0.8,
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8) if trial is not None else 2,
        }
        model = GradientBoostingRegressor(**params, random_state=RANDOM_STATE)

    elif model_name == "XGBoost":
        if not HAS_XGBOOST:
            raise ImportError("xgboost is not installed.")
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 80, 700) if trial is not None else 300,
            "max_depth": trial.suggest_int("max_depth", 1, 5) if trial is not None else 2,
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True) if trial is not None else 0.03,
            "subsample": trial.suggest_float("subsample", 0.5, 1.0) if trial is not None else 0.8,
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0) if trial is not None else 0.8,
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 100.0, log=True) if trial is not None else 1.0,
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-6, 10.0, log=True) if trial is not None else 0.0,
        }
        model = XGBRegressor(**params, objective="reg:squarederror", random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)

    elif model_name == "LightGBM":
        if not HAS_LIGHTGBM:
            raise ImportError("lightgbm is not installed.")
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 80, 700) if trial is not None else 300,
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True) if trial is not None else 0.03,
            "num_leaves": trial.suggest_int("num_leaves", 4, 40) if trial is not None else 15,
            "max_depth": trial.suggest_categorical("max_depth", [-1, 2, 3, 4, 5, 6]) if trial is not None else -1,
            "subsample": trial.suggest_float("subsample", 0.5, 1.0) if trial is not None else 0.8,
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0) if trial is not None else 0.8,
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 100.0, log=True) if trial is not None else 1.0,
            "min_child_samples": trial.suggest_int("min_child_samples", 2, 20) if trial is not None else 5,
        }
        model = LGBMRegressor(**params, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)

    elif model_name == "CatBoost":
        if not HAS_CATBOOST:
            raise ImportError("catboost is not installed.")
        params = {
            "iterations": trial.suggest_int("iterations", 100, 800) if trial is not None else 400,
            "depth": trial.suggest_int("depth", 2, 6) if trial is not None else 3,
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True) if trial is not None else 0.03,
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 30.0, log=True) if trial is not None else 3.0,
        }
        model = CatBoostRegressor(**params, loss_function="RMSE", random_seed=RANDOM_STATE, verbose=False,
                                  allow_writing_files=False)

    elif model_name == "GaussianProcess":
        alpha = trial.suggest_float("alpha", 1e-8, 1e-1, log=True) if trial is not None else 1e-3
        length_scale = trial.suggest_float("length_scale", 1e-2, 1e2, log=True) if trial is not None else 1.0
        noise_level = trial.suggest_float("noise_level", 1e-6, 1.0, log=True) if trial is not None else 1e-2
        kernel = ConstantKernel(1.0) * RBF(length_scale=length_scale) + WhiteKernel(noise_level=noise_level)
        model = GaussianProcessRegressor(kernel=kernel, alpha=alpha, normalize_y=True, random_state=RANDOM_STATE)

    elif model_name == "KNN":
        max_neighbors = max(2, min(20, n_train_samples - 1))
        n_neighbors = trial.suggest_int("n_neighbors", 2, max_neighbors) if trial is not None else min(5, max_neighbors)
        weights = trial.suggest_categorical("weights", ["uniform", "distance"]) if trial is not None else "distance"
        p = trial.suggest_int("p", 1, 2) if trial is not None else 2
        model = KNeighborsRegressor(n_neighbors=n_neighbors, weights=weights, p=p)

    elif model_name == "AdaBoost":
        max_depth = trial.suggest_int("max_depth", 1, 5) if trial is not None else 2
        n_estimators = trial.suggest_int("n_estimators", 50, 600) if trial is not None else 200
        learning_rate = trial.suggest_float("learning_rate", 0.005, 0.5, log=True) if trial is not None else 0.05
        base = DecisionTreeRegressor(max_depth=max_depth, random_state=RANDOM_STATE)
        model = build_adaboost(base, n_estimators, learning_rate)

    else:
        raise ValueError(f"Unknown model name: {model_name}")

    return wrap_with_scaler_if_needed(model_name, model)


ALL_MODEL_NAMES = [
    "MLR", "PLS", "Ridge", "LASSO", "ElasticNet", "SVR", "RandomForest",
    "ExtraTrees", "GradientBoosting", "XGBoost", "LightGBM", "CatBoost",
    "GaussianProcess", "KNN", "AdaBoost"
]
AVAILABLE_MODEL_NAMES = []
for name in ALL_MODEL_NAMES:
    if model_is_available(name):
        AVAILABLE_MODEL_NAMES.append(name)
    else:
        add_warning(f"{name} is unavailable and will be skipped.")

print("Available models:")
print(AVAILABLE_MODEL_NAMES)


Available models:
['MLR', 'PLS', 'Ridge', 'LASSO', 'ElasticNet', 'SVR', 'RandomForest', 'ExtraTrees', 'GradientBoosting', 'XGBoost', 'LightGBM', 'CatBoost', 'GaussianProcess', 'KNN', 'AdaBoost']


## 8. Candidate feature subsets and aggregated feature ranking

Several alternative feature subsets are generated from the preselected descriptor pool rather than relying on a single feature-selection method.

The candidate subsets include:

- all descriptors remaining after preprocessing;
- LASSO-selected descriptors;
- RFE-selected descriptors;
- descriptors ranked by permutation importance;
- descriptors ranked by SHAP importance;
- aggregated rank-based subsets containing the top 5, 8, 10, 12, 15, 20, 25, and 30 descriptors.

The aggregated ranking combines LASSO, RFE, permutation-importance, and SHAP ranks. All feature-selection procedures use the training data only.

The resulting feature rankings are stored in `fs_lasso`, `fs_rfe`, `fs_perm`, `fs_shap`, and `rank_df`, and the candidate subsets are collected in `candidate_subsets`.


In [9]:
# ============================================================
# Feature ranking and candidate subset generation
# ============================================================


def lasso_feature_ranking(X, y):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    cv_splits = make_complex_only_cv_splits(train_meta, n_splits=N_SPLITS_CV, n_repeats=1, random_state=RANDOM_STATE)
    lasso_cv = LassoCV(alphas=np.logspace(-5, 1, 120), cv=cv_splits, max_iter=300000, random_state=RANDOM_STATE)
    lasso_cv.fit(X_scaled, y)
    coef = np.asarray(lasso_cv.coef_).reshape(-1)
    abs_coef = np.abs(coef)
    df_lasso = pd.DataFrame({
        "feature": X.columns,
        "coefficient": coef,
        "abs_coefficient": abs_coef,
        "selected_nonzero": abs_coef > 1e-10,
        "alpha_selected": lasso_cv.alpha_
    }).sort_values("abs_coefficient", ascending=False)
    df_lasso["lasso_rank"] = np.arange(1, len(df_lasso) + 1)
    subset = df_lasso.loc[df_lasso["selected_nonzero"], "feature"].tolist()
    if len(subset) == 0:
        subset = df_lasso.head(min(10, X.shape[1]))["feature"].tolist()
        add_warning("LASSO selected zero non-zero coefficients. Fallback: top 10 absolute coefficients.")
    return subset, df_lasso


def rfe_feature_ranking(X, y):
    estimator = Ridge(alpha=1.0, random_state=RANDOM_STATE)
    X_scaled = StandardScaler().fit_transform(X)
    n_features_to_select = max(3, min(20, X.shape[1] // 2 if X.shape[1] > 6 else X.shape[1]))
    rfe = RFE(estimator=estimator, n_features_to_select=n_features_to_select, step=1)
    rfe.fit(X_scaled, y)
    df_rfe = pd.DataFrame({"feature": X.columns, "rfe_rank": rfe.ranking_, "selected": rfe.support_}).sort_values(
        ["rfe_rank", "feature"])
    subset = df_rfe.loc[df_rfe["selected"], "feature"].tolist()
    return subset, df_rfe


def permutation_feature_ranking(X, y):
    base_model = ExtraTreesRegressor(n_estimators=700, random_state=RANDOM_STATE, n_jobs=-1, max_features=1.0)
    base_model.fit(X, y)
    perm = permutation_importance(base_model, X, y, n_repeats=N_PERMUTATION_REPEATS, random_state=RANDOM_STATE,
                                  n_jobs=-1, scoring="neg_root_mean_squared_error")
    df_perm = pd.DataFrame({
        "feature": X.columns,
        "permutation_importance_mean": perm.importances_mean,
        "permutation_importance_std": perm.importances_std
    }).sort_values("permutation_importance_mean", ascending=False)
    df_perm["permutation_rank"] = np.arange(1, len(df_perm) + 1)
    subset = df_perm.head(min(20, X.shape[1]))["feature"].tolist()
    return subset, df_perm


def shap_feature_ranking(X, y):
    model = ExtraTreesRegressor(n_estimators=700, random_state=RANDOM_STATE, n_jobs=-1, max_features=1.0)
    model.fit(X, y)
    if HAS_SHAP:
        try:
            explainer = shap.TreeExplainer(model)
            shap_values = np.asarray(explainer.shap_values(X))
            if shap_values.ndim == 3:
                shap_values = shap_values[:, :, 0]
            mean_abs = np.mean(np.abs(shap_values), axis=0)
        except Exception as e:
            add_warning(f"SHAP feature ranking failed; fallback to ExtraTrees importances: {e}")
            mean_abs = model.feature_importances_
    else:
        add_warning("SHAP is unavailable. shap_subset falls back to ExtraTrees feature_importances_.")
        mean_abs = model.feature_importances_
    df_shap = pd.DataFrame({"feature": X.columns, "mean_abs_shap": mean_abs}).sort_values("mean_abs_shap",
                                                                                          ascending=False)
    df_shap["shap_rank"] = np.arange(1, len(df_shap) + 1)
    subset = df_shap.head(min(20, X.shape[1]))["feature"].tolist()
    return subset, df_shap


X_fs = X_train_final_base.copy()
y_fs = y_train.copy()
all_features = list(X_fs.columns)

lasso_subset, fs_lasso = lasso_feature_ranking(X_fs, y_fs)
rfe_subset, fs_rfe = rfe_feature_ranking(X_fs, y_fs)
permutation_subset, fs_perm = permutation_feature_ranking(X_fs, y_fs)
shap_subset, fs_shap = shap_feature_ranking(X_fs, y_fs)

TABLES["feature_selection_lasso"] = fs_lasso
TABLES["feature_selection_rfe"] = fs_rfe
TABLES["feature_selection_permutation"] = fs_perm
TABLES["feature_selection_shap"] = fs_shap

rank_df = pd.DataFrame({"feature": all_features})
rank_df = rank_df.merge(fs_lasso[["feature", "lasso_rank"]], on="feature", how="left")
rank_df = rank_df.merge(fs_rfe[["feature", "rfe_rank"]], on="feature", how="left")
rank_df = rank_df.merge(fs_perm[["feature", "permutation_rank"]], on="feature", how="left")
rank_df = rank_df.merge(fs_shap[["feature", "shap_rank"]], on="feature", how="left")
for col in ["lasso_rank", "rfe_rank", "permutation_rank", "shap_rank"]:
    rank_df[col] = rank_df[col].fillna(rank_df[col].max() + 1)
rank_df["mean_rank"] = rank_df[["lasso_rank", "rfe_rank", "permutation_rank", "shap_rank"]].mean(axis=1)
rank_df["median_rank"] = rank_df[["lasso_rank", "rfe_rank", "permutation_rank", "shap_rank"]].median(axis=1)
rank_df = rank_df.sort_values(["mean_rank", "median_rank", "feature"])
rank_df["aggregated_rank"] = np.arange(1, len(rank_df) + 1)
TABLES["feature_rank_aggregation"] = rank_df

candidate_subsets = {
    "all_62_or_cleaned_all": all_features,
    "lasso_subset": [f for f in lasso_subset if f in all_features],
    "rfe_subset": [f for f in rfe_subset if f in all_features],
    "permutation_subset": [f for f in permutation_subset if f in all_features],
    "shap_subset": [f for f in shap_subset if f in all_features]
}
for n in [5, 8, 10, 12, 15, 20, 25, 30]:
    candidate_subsets[f"rank_top_{n}"] = rank_df.head(min(n, len(all_features)))["feature"].tolist()

clean_candidate_subsets = {}
seen_signatures = set()
for subset_name, features in candidate_subsets.items():
    features = [f for f in features if f in all_features]
    if len(features) == 0:
        continue
    signature = tuple(features)
    if signature not in seen_signatures:
        clean_candidate_subsets[subset_name] = features
        seen_signatures.add(signature)
candidate_subsets = clean_candidate_subsets

candidate_summary = pd.DataFrame([{
    "subset_name": name,
    "n_features": len(features),
    "features": "; ".join(features)
} for name, features in candidate_subsets.items()])
save_dataframe(candidate_summary, "candidate_feature_subsets.xlsx")
save_dataframe(rank_df, "feature_rank_aggregation.xlsx")

fig, ax = plt.subplots(figsize=(9, max(5, 0.25 * min(40, len(rank_df)))))
plot_rank = rank_df.sort_values("aggregated_rank").head(40).sort_values("aggregated_rank", ascending=False)
ax.barh(plot_rank["feature"], plot_rank["mean_rank"])
ax.set_xlabel("Mean rank, lower is better")
ax.set_ylabel("Feature")
ax.set_title("Aggregated feature ranking")
save_figure(fig, "feature_rank_aggregation_top40")

print(f"Candidate subsets: {len(candidate_subsets)}")
display(candidate_summary)


Candidate subsets: 12


,subset_name,n_features,features
0,all_62_or_cleaned_all,25,DLS_cons:(alvaDesc); F05[N-X]:(alvaDesc); MATS...
1,lasso_subset,23,Koopmans_EA; Dipole_moment; RDF060u:(alvaDesc)...
2,rfe_subset,12,DLS_cons:(alvaDesc); Dipole_moment; F05[N-X]:(...
3,permutation_subset,20,RDF060e:(alvaDesc); Se1Cu2Ag1s:(OEstate); Koop...
4,shap_subset,20,RDF060e:(alvaDesc); Se1Cu2Ag1s:(OEstate); Koop...
5,rank_top_5,5,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...
6,rank_top_8,8,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...
7,rank_top_10,10,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...
8,rank_top_12,12,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...
9,rank_top_15,15,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...


## 9. Selection of global and model-specific feature subsets

Each candidate subset is evaluated by repeated complex-only cross-validation using several selector models: Ridge, SVR, Extra Trees, Random Forest, CatBoost, XGBoost, and LightGBM when available.

For each subset–model combination, mean and standard deviation of Q², RMSE, and MAE are calculated. These results are used to identify:

- `GLOBAL_ROBUST_SUBSET_NAME` — the subset with the best overall performance across selector models;
- `MODEL_SPECIFIC_BEST_SUBSETS` — the best-performing subset for each individual model family.

The complete comparison is stored in `feature_subset_comparison`.


In [10]:
# ============================================================
# Global and model-specific subset selection
# ============================================================

selector_models_requested = ["Ridge", "SVR", "ExtraTrees", "RandomForest", "CatBoost", "XGBoost", "LightGBM"]
selector_models = [m for m in selector_models_requested if model_is_available(m)]
for m in selector_models_requested:
    if m not in selector_models:
        add_warning(f"Selector model {m} is unavailable and skipped.")

subset_eval_records = []
for subset_name, features in candidate_subsets.items():
    X_subset = X_train_final_base[features]
    for model_name in selector_models:
        try:
            estimator = build_estimator(model_name, trial=None, n_features=len(features), n_train_samples=len(y_train))
            cv_summary, _ = evaluate_cv_model(estimator, X_subset, y_train, train_meta, n_splits=N_SPLITS_CV,
                                              n_repeats=N_REPEATS_CV, random_state=RANDOM_STATE)
            record = {"subset_name": subset_name, "selector_model": model_name, "n_features": len(features),
                      "features": "; ".join(features)}
            record.update(cv_summary)
            subset_eval_records.append(record)
        except Exception as e:
            subset_eval_records.append({
                "subset_name": subset_name, "selector_model": model_name, "n_features": len(features),
                "features": "; ".join(features), "Q2_mean": np.nan, "Q2_std": np.nan,
                "RMSE_CV_mean": np.nan, "RMSE_CV_std": np.nan, "MAE_CV_mean": np.nan,
                "MAE_CV_std": np.nan, "n_cv_folds": 0, "n_failed_folds": np.nan, "error": str(e)
            })
            add_warning(f"Subset evaluation failed for {subset_name} + {model_name}: {e}")

feature_subset_comparison = pd.DataFrame(subset_eval_records)
ranked_subset_comparison = feature_subset_comparison.copy()
ranked_subset_comparison["rank_Q2"] = ranked_subset_comparison.groupby("selector_model")["Q2_mean"].rank(
    ascending=False, method="average", na_option="bottom")
ranked_subset_comparison["rank_RMSE"] = ranked_subset_comparison.groupby("selector_model")["RMSE_CV_mean"].rank(
    ascending=True, method="average", na_option="bottom")
ranked_subset_comparison["rank_Q2_std"] = ranked_subset_comparison.groupby("selector_model")["Q2_std"].rank(
    ascending=True, method="average", na_option="bottom")
ranked_subset_comparison["rank_n_features"] = ranked_subset_comparison.groupby("selector_model")["n_features"].rank(
    ascending=True, method="average", na_option="bottom")
ranked_subset_comparison["aggregated_subset_rank_score"] = (
        ranked_subset_comparison["rank_Q2"] * 0.40 +
        ranked_subset_comparison["rank_RMSE"] * 0.30 +
        ranked_subset_comparison["rank_Q2_std"] * 0.20 +
        ranked_subset_comparison["rank_n_features"] * 0.10
)
TABLES["feature_subset_comparison"] = ranked_subset_comparison

global_subset_rank = ranked_subset_comparison.groupby("subset_name", as_index=False).agg(
    n_features=("n_features", "first"),
    mean_Q2=("Q2_mean", "mean"),
    mean_RMSE=("RMSE_CV_mean", "mean"),
    mean_Q2_std=("Q2_std", "mean"),
    mean_rank_score=("aggregated_subset_rank_score", "mean")
)
global_subset_rank["global_rank_Q2"] = global_subset_rank["mean_Q2"].rank(ascending=False, na_option="bottom")
global_subset_rank["global_rank_RMSE"] = global_subset_rank["mean_RMSE"].rank(ascending=True, na_option="bottom")
global_subset_rank["global_rank_Q2_std"] = global_subset_rank["mean_Q2_std"].rank(ascending=True, na_option="bottom")
global_subset_rank["global_rank_n_features"] = global_subset_rank["n_features"].rank(ascending=True, na_option="bottom")
global_subset_rank["global_robust_score"] = (
        global_subset_rank["global_rank_Q2"] * 0.40 +
        global_subset_rank["global_rank_RMSE"] * 0.30 +
        global_subset_rank["global_rank_Q2_std"] * 0.20 +
        global_subset_rank["global_rank_n_features"] * 0.10
)
global_subset_rank = global_subset_rank.sort_values("global_robust_score")
GLOBAL_ROBUST_SUBSET_NAME = global_subset_rank.iloc[0]["subset_name"]
GLOBAL_ROBUST_FEATURES = candidate_subsets[GLOBAL_ROBUST_SUBSET_NAME]

MODEL_SPECIFIC_BEST_SUBSETS = {}
model_specific_records = []
for model_name in AVAILABLE_MODEL_NAMES:
    if model_name in ranked_subset_comparison["selector_model"].unique():
        tmp = ranked_subset_comparison[ranked_subset_comparison["selector_model"] == model_name].copy()
    else:
        tmp = ranked_subset_comparison.groupby("subset_name", as_index=False).agg(
            n_features=("n_features", "first"), Q2_mean=("Q2_mean", "mean"), Q2_std=("Q2_std", "mean"),
            RMSE_CV_mean=("RMSE_CV_mean", "mean"), RMSE_CV_std=("RMSE_CV_std", "mean"),
            MAE_CV_mean=("MAE_CV_mean", "mean"), MAE_CV_std=("MAE_CV_std", "mean"),
            aggregated_subset_rank_score=("aggregated_subset_rank_score", "mean"), features=("features", "first")
        )
    tmp = tmp.sort_values(["aggregated_subset_rank_score", "RMSE_CV_mean", "Q2_std", "n_features"],
                          ascending=[True, True, True, True])
    best_subset_name = tmp.iloc[0]["subset_name"]
    MODEL_SPECIFIC_BEST_SUBSETS[model_name] = {"subset_name": best_subset_name,
                                               "features": candidate_subsets[best_subset_name]}
    model_specific_records.append({
        "model_name": model_name,
        "best_subset_name": best_subset_name,
        "n_features": len(candidate_subsets[best_subset_name]),
        "features": "; ".join(candidate_subsets[best_subset_name]),
        "basis": "model-specific selector CV" if model_name in ranked_subset_comparison[
            "selector_model"].unique() else "global robust fallback"
    })

model_specific_best_subsets = pd.DataFrame(model_specific_records)
TABLES["model_specific_best_subsets"] = model_specific_best_subsets
save_dataframe(ranked_subset_comparison, "feature_subset_comparison.xlsx")
save_dataframe(model_specific_best_subsets, "model_specific_best_subsets.xlsx")
save_dataframe(global_subset_rank, "global_subset_rank.xlsx")

fig, ax = plt.subplots(figsize=(9, 5))
plot_df = global_subset_rank.sort_values("global_robust_score", ascending=False)
ax.barh(plot_df["subset_name"], plot_df["global_robust_score"])
ax.set_xlabel("Global robust subset score, lower is better")
ax.set_ylabel("Feature subset")
ax.set_title("Feature subset comparison")
save_figure(fig, "feature_subset_comparison_global")

print(f"Global robust subset: {GLOBAL_ROBUST_SUBSET_NAME}, n_features={len(GLOBAL_ROBUST_FEATURES)}")
display(global_subset_rank)
display(model_specific_best_subsets)


Global robust subset: rank_top_8, n_features=8


,subset_name,n_features,mean_Q2,mean_RMSE,mean_Q2_std,mean_rank_score,global_rank_Q2,global_rank_RMSE,global_rank_Q2_std,global_rank_n_features,global_robust_score
9,rank_top_8,8,0.51,0.04,0.30,3.12,1.00,1.00,3.00,2.00,1.50
4,rank_top_12,12,0.51,0.04,0.26,3.61,2.00,2.00,1.00,4.50,2.05
3,rank_top_10,10,0.49,0.04,0.29,5.05,3.00,3.00,2.00,3.00,2.80
5,rank_top_15,15,0.47,0.04,0.31,5.34,4.00,4.00,4.00,6.00,4.20
6,rank_top_20,20,0.43,0.04,0.40,6.16,5.00,5.00,5.00,8.00,5.30
8,rank_top_5,5,0.40,0.04,0.41,5.35,6.00,6.00,6.00,1.00,5.50
11,shap_subset,20,0.38,0.04,0.41,6.71,7.00,7.00,8.00,8.00,7.30
2,permutation_subset,20,0.37,0.04,0.41,7.75,8.00,9.00,7.00,8.00,8.10
1,lasso_subset,23,0.36,0.04,0.47,8.54,9.00,8.00,9.00,10.00,8.80
7,rank_top_25,25,0.34,0.04,0.51,8.61,10.00,10.00,11.00,11.50,10.35


,model_name,best_subset_name,n_features,features,basis
0,MLR,rank_top_8,8,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,global robust fallback
1,PLS,rank_top_8,8,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,global robust fallback
2,Ridge,rank_top_20,20,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,model-specific selector CV
3,LASSO,rank_top_8,8,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,global robust fallback
4,ElasticNet,rank_top_8,8,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,global robust fallback
5,SVR,rank_top_10,10,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,model-specific selector CV
6,RandomForest,rank_top_8,8,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,model-specific selector CV
7,ExtraTrees,rank_top_8,8,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,model-specific selector CV
8,GradientBoosting,rank_top_8,8,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,global robust fallback
9,XGBoost,rank_top_8,8,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,model-specific selector CV


## 10. Preliminary LazyPredict screening

When `lazypredict` is available, `LazyRegressor` is run on the global robust subset as an exploratory model-family screen.

These results are saved separately and are not used as the primary basis for final model selection. The main ranking is based on the controlled validation and model-evaluation procedures implemented in the subsequent sections.


In [11]:
# ============================================================
# LazyPredict screening
# ============================================================

if HAS_LAZYPREDICT:
    try:
        lazy_reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=False,
                                 random_state=RANDOM_STATE)
        lazy_models, _ = lazy_reg.fit(
            X_train_final_base[GLOBAL_ROBUST_FEATURES],
            X_test_final_base[GLOBAL_ROBUST_FEATURES],
            y_train,
            y_test
        )
        lazy_results = lazy_models.reset_index().rename(columns={"index": "LazyPredict_model"})
        save_dataframe(lazy_results, "lazypredict_screening.xlsx")
        TABLES["lazypredict_screening"] = lazy_results
        display(lazy_results.head(20))
    except Exception as e:
        add_warning(f"LazyPredict screening failed: {e}")
        TABLES["lazypredict_screening"] = pd.DataFrame({"warning": [str(e)]})
else:
    add_warning("LazyPredict is not installed. LazyPredict screening skipped.")
    TABLES["lazypredict_screening"] = pd.DataFrame({"warning": ["LazyPredict is not installed."]})


  0%|          | 0/42 [00:00<?, ?it/s]

,Model,Adjusted R-Squared,R-Squared,RMSE,Time Taken
0,GradientBoostingRegressor,0.74,0.85,0.04,0.05
1,NuSVR,0.72,0.84,0.04,0.01
2,HistGradientBoostingRegressor,0.69,0.82,0.04,0.06
3,LGBMRegressor,0.63,0.78,0.04,0.02
4,ExtraTreesRegressor,0.57,0.75,0.05,0.08
5,LinearRegression,0.48,0.70,0.05,0.01
6,TransformedTargetRegressor,0.48,0.70,0.05,0.01
7,LassoLarsCV,0.48,0.70,0.05,0.02
8,Lars,0.48,0.70,0.05,0.01
9,LassoLarsIC,0.48,0.70,0.05,0.01


## 11. Hyperparameter optimization and model screening

All available model families are evaluated using the global robust subset and, where different, the corresponding model-specific subset.

Hyperparameters are optimized with Optuna using repeated complex-only cross-validation on the training/internal dataset. The objective minimizes cross-validated RMSE with an additional penalty for variability across folds.

After optimization, each model is re-evaluated using the full repeated cross-validation scheme, fitted to the complete training set, and evaluated on the fixed external test set.

Fitted models, Optuna trial histories, performance metrics, feature configurations, and train/test predictions are saved for subsequent ranking and validation.


In [12]:
# ============================================================
# Hyperparameter optimization and model evaluation
# ============================================================


def optimize_model_with_optuna(model_name, X, y, meta, n_trials=40):
    if not HAS_OPTUNA:
        add_warning(f"Optuna is unavailable. {model_name} uses default parameters.")
        estimator = build_estimator(model_name, trial=None, n_features=X.shape[1], n_train_samples=len(y))
        return estimator, {"optimization": "default_no_optuna"}, pd.DataFrame()

    def objective(trial):
        try:
            estimator = build_estimator(model_name, trial=trial, n_features=X.shape[1], n_train_samples=len(y))
            cv_summary, _ = evaluate_cv_model(
                estimator, X, y, meta,
                n_splits=N_SPLITS_CV,
                n_repeats=N_OPTUNA_CV_REPEATS,
                random_state=RANDOM_STATE
            )
            cv_rmse = cv_summary["RMSE_CV_mean"]
            q2_std = cv_summary["Q2_std"]
            if pd.isna(cv_rmse):
                return 1e9
            penalty = 0.0 if pd.isna(q2_std) else 0.05 * q2_std
            return float(cv_rmse + penalty)
        except Exception:
            return 1e9

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best_params = study.best_params
    fixed_trial = optuna.trial.FixedTrial(best_params)
    best_estimator = build_estimator(model_name, trial=fixed_trial, n_features=X.shape[1], n_train_samples=len(y))
    trials_df = study.trials_dataframe()
    return best_estimator, best_params, trials_df


all_model_records = []
all_predictions_records = []
trained_models = {}
model_feature_sets = {}
model_configs = {}

for model_name in AVAILABLE_MODEL_NAMES:
    modes = [{"feature_mode": "global_robust_subset", "subset_name": GLOBAL_ROBUST_SUBSET_NAME,
              "features": GLOBAL_ROBUST_FEATURES}]
    specific_subset_name = MODEL_SPECIFIC_BEST_SUBSETS[model_name]["subset_name"]
    specific_features = MODEL_SPECIFIC_BEST_SUBSETS[model_name]["features"]
    if specific_subset_name != GLOBAL_ROBUST_SUBSET_NAME:
        modes.append({"feature_mode": "model_specific_best_subset", "subset_name": specific_subset_name,
                      "features": specific_features})

    for mode in modes:
        feature_mode = mode["feature_mode"]
        subset_name = mode["subset_name"]
        features = mode["features"]
        X_train_model = X_train_final_base[features]
        X_test_model = X_test_final_base[features]
        model_key = f"{model_name}__{feature_mode}__{subset_name}"
        print(f"Optimizing/evaluating {model_key} | p={len(features)}")
        try:
            best_estimator, best_params, trials_df = optimize_model_with_optuna(model_name, X_train_model, y_train,
                                                                                train_meta, n_trials=N_OPTUNA_TRIALS)
            cv_summary, cv_folds = evaluate_cv_model(best_estimator, X_train_model, y_train, train_meta,
                                                     n_splits=N_SPLITS_CV, n_repeats=N_REPEATS_CV,
                                                     random_state=RANDOM_STATE)
            fitted_model, train_test_metrics, predictions = evaluate_train_test(best_estimator, X_train_model, y_train,
                                                                                X_test_model, y_test, p=len(features))

            trained_models[model_key] = fitted_model
            model_feature_sets[model_key] = features
            model_configs[model_key] = {"model_name": model_name, "feature_mode": feature_mode,
                                        "subset_name": subset_name, "best_params": best_params, "features": features}

            model_path = versioned_path(MODEL_DIR / f"{re.sub(r'[^A-Za-z0-9_\-\.]+', '_', model_key)}.joblib",
                                        OVERWRITE_EXISTING_RESULTS)
            joblib.dump({"model": fitted_model, "features": features, "config": model_configs[model_key]}, model_path)

            if not trials_df.empty:
                trials_path = versioned_path(
                    TABLE_DIR / f"optuna_trials_{re.sub(r'[^A-Za-z0-9_\-\.]+', '_', model_key)}.xlsx",
                    OVERWRITE_EXISTING_RESULTS)
                trials_df.to_excel(trials_path, index=False)

            record = {
                "model_key": model_key,
                "model_name": model_name,
                "feature_mode": feature_mode,
                "subset_name": subset_name,
                "n_features": len(features),
                "features": "; ".join(features),
                "best_params_json": json.dumps(best_params, ensure_ascii=False),
                "model_path": str(model_path)
            }
            record.update(cv_summary)
            record.update(train_test_metrics)
            all_model_records.append(record)

            predictions["model_key"] = model_key
            predictions["model_name"] = model_name
            predictions["feature_mode"] = feature_mode
            predictions["subset_name"] = subset_name
            all_predictions_records.append(predictions)

        except Exception as e:
            add_warning(f"Model screening failed for {model_key}: {e}")
            all_model_records.append({
                "model_key": model_key, "model_name": model_name, "feature_mode": feature_mode,
                "subset_name": subset_name, "n_features": len(features), "features": "; ".join(features),
                "error": str(e)
            })

all_model_metrics = pd.DataFrame(all_model_records)
all_predictions = pd.concat(all_predictions_records, axis=0,
                            ignore_index=True) if all_predictions_records else pd.DataFrame()
TABLES["all_model_metrics"] = all_model_metrics
TABLES["all_predictions"] = all_predictions
save_dataframe(all_model_metrics, "all_model_metrics.xlsx")
save_dataframe(all_predictions, "all_model_predictions.xlsx")

if len(trained_models) == 0:
    raise RuntimeError("No models were successfully trained.")

display(all_model_metrics.sort_values(["Q2_mean", "RMSE_test"], ascending=[False, True]).head(20))


Optimizing/evaluating MLR__global_robust_subset__rank_top_8 | p=8
Optimizing/evaluating PLS__global_robust_subset__rank_top_8 | p=8
Optimizing/evaluating Ridge__global_robust_subset__rank_top_8 | p=8
Optimizing/evaluating Ridge__model_specific_best_subset__rank_top_20 | p=20
Optimizing/evaluating LASSO__global_robust_subset__rank_top_8 | p=8
Optimizing/evaluating ElasticNet__global_robust_subset__rank_top_8 | p=8
Optimizing/evaluating SVR__global_robust_subset__rank_top_8 | p=8
Optimizing/evaluating SVR__model_specific_best_subset__rank_top_10 | p=10
Optimizing/evaluating RandomForest__global_robust_subset__rank_top_8 | p=8
Optimizing/evaluating ExtraTrees__global_robust_subset__rank_top_8 | p=8
Optimizing/evaluating GradientBoosting__global_robust_subset__rank_top_8 | p=8
Optimizing/evaluating XGBoost__global_robust_subset__rank_top_8 | p=8
Optimizing/evaluating LightGBM__global_robust_subset__rank_top_8 | p=8
Optimizing/evaluating LightGBM__model_specific_best_subset__rank_top_5 | p=

,model_key,model_name,feature_mode,subset_name,n_features,features,best_params_json,model_path,Q2_mean,Q2_std,...,Q2F3,GT_r2,GT_r0_squared,GT_r0prime_squared,GT_k,GT_kprime,GT_abs_r2_minus_r0sq_over_r2,GT_abs_r2_minus_r0psq_over_r2,GT_criteria_pass,error
7,SVR__model_specific_best_subset__rank_top_10,SVR,model_specific_best_subset,rank_top_10,10,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,"{""C"": 7.63561552176222, ""epsilon"": 0.000279821...",results_QSPR_QSAR_LogS1_20260702_13\models\SVR...,0.77,0.15,...,0.87,0.91,0.92,0.88,1.00,1.00,0.01,0.03,True,NaN
6,SVR__global_robust_subset__rank_top_8,SVR,global_robust_subset,rank_top_8,8,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,"{""C"": 609.1394159031862, ""epsilon"": 0.00131839...",results_QSPR_QSAR_LogS1_20260702_13\models\SVR...,0.75,0.17,...,0.82,0.88,0.90,0.84,1.00,1.00,0.02,0.04,True,NaN
14,CatBoost__global_robust_subset__rank_top_8,CatBoost,global_robust_subset,rank_top_8,8,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,"{""iterations"": 715, ""depth"": 3, ""learning_rate...",results_QSPR_QSAR_LogS1_20260702_13\models\Cat...,0.72,0.19,...,0.86,0.91,0.91,0.85,1.00,1.00,0.01,0.06,True,NaN
10,GradientBoosting__global_robust_subset__rank_t...,GradientBoosting,global_robust_subset,rank_top_8,8,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,"{""n_estimators"": 663, ""learning_rate"": 0.06362...",results_QSPR_QSAR_LogS1_20260702_13\models\Gra...,0.71,0.18,...,0.79,0.85,0.87,0.72,1.00,1.00,0.02,0.15,True,NaN
15,GaussianProcess__global_robust_subset__rank_top_8,GaussianProcess,global_robust_subset,rank_top_8,8,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,"{""alpha"": 2.3144513393908483e-07, ""length_scal...",results_QSPR_QSAR_LogS1_20260702_13\models\Gau...,0.71,0.17,...,0.89,0.93,0.93,0.93,1.00,1.00,0.01,0.00,True,NaN
11,XGBoost__global_robust_subset__rank_top_8,XGBoost,global_robust_subset,rank_top_8,8,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,"{""n_estimators"": 652, ""max_depth"": 3, ""learnin...",results_QSPR_QSAR_LogS1_20260702_13\models\XGB...,0.66,0.21,...,0.66,0.76,0.80,0.44,1.01,0.99,0.05,0.43,True,NaN
17,AdaBoost__global_robust_subset__rank_top_8,AdaBoost,global_robust_subset,rank_top_8,8,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,"{""max_depth"": 5, ""n_estimators"": 483, ""learnin...",results_QSPR_QSAR_LogS1_20260702_13\models\Ada...,0.60,0.28,...,0.40,0.59,0.65,-0.64,1.01,0.99,0.11,2.09,False,NaN
3,Ridge__model_specific_best_subset__rank_top_20,Ridge,model_specific_best_subset,rank_top_20,20,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,"{""alpha"": 27.035686393714283}",results_QSPR_QSAR_LogS1_20260702_13\models\Rid...,0.58,0.22,...,0.48,0.64,0.67,-0.34,1.01,0.99,0.04,1.53,True,NaN
9,ExtraTrees__global_robust_subset__rank_top_8,ExtraTrees,global_robust_subset,rank_top_8,8,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,"{""n_estimators"": 865, ""max_depth"": 8, ""min_sam...",results_QSPR_QSAR_LogS1_20260702_13\models\Ext...,0.58,0.16,...,0.43,0.61,0.65,-0.72,1.01,0.99,0.08,2.18,True,NaN
8,RandomForest__global_robust_subset__rank_top_8,RandomForest,global_robust_subset,rank_top_8,8,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,"{""n_estimators"": 456, ""max_depth"": 8, ""min_sam...",results_QSPR_QSAR_LogS1_20260702_13\models\Ran...,0.55,0.19,...,0.36,0.55,0.60,-1.39,1.01,0.99,0.08,3.51,False,NaN


## 12. Model ranking, applicability domain, and Y-randomization

Successfully fitted models are first ranked using repeated-CV performance, external-test metrics, concordance statistics, Q²F metrics, and model size.

The highest-ranked preliminary candidates are then subjected to applicability-domain analysis and Y-randomization. Applicability-domain assessment combines leverage and standardized residuals, using the leverage threshold \(h^* = 3(p+1)/n\) and a standardized-residual threshold of ±3.

Y-randomization compares the observed repeated-CV Q² with the distribution obtained after random permutation of the target values.

These diagnostics are incorporated into the final robust ranking, from which the top three models are selected.


In [13]:
# ============================================================
# Final ranking, applicability domain, and Y-randomization
# ============================================================


def add_rank_column(df, source_col, rank_col, ascending):
    df[rank_col] = df[source_col].rank(ascending=ascending, method="average", na_option="bottom")
    return df


def preliminary_rank_models(metrics_df):
    ranking = metrics_df[metrics_df["model_key"].isin(trained_models.keys())].copy()
    ranking = add_rank_column(ranking, "Q2_mean", "rank_CV_Q2", ascending=False)
    ranking = add_rank_column(ranking, "RMSE_CV_mean", "rank_CV_RMSE", ascending=True)
    ranking = add_rank_column(ranking, "Q2_std", "rank_CV_Q2_std", ascending=True)
    ranking = add_rank_column(ranking, "R2_test", "rank_test_R2", ascending=False)
    ranking = add_rank_column(ranking, "RMSE_test", "rank_test_RMSE", ascending=True)
    ranking = add_rank_column(ranking, "CCC", "rank_CCC", ascending=False)
    ranking = add_rank_column(ranking, "Q2F1", "rank_Q2F1", ascending=False)
    ranking = add_rank_column(ranking, "Q2F2", "rank_Q2F2", ascending=False)
    ranking = add_rank_column(ranking, "Q2F3", "rank_Q2F3", ascending=False)
    ranking = add_rank_column(ranking, "n_features", "rank_n_features", ascending=True)
    ranking["preliminary_rank_score"] = (
            ranking["rank_CV_Q2"] * 0.22 +
            ranking["rank_CV_RMSE"] * 0.18 +
            ranking["rank_CV_Q2_std"] * 0.10 +
            ranking["rank_test_R2"] * 0.16 +
            ranking["rank_test_RMSE"] * 0.14 +
            ranking["rank_CCC"] * 0.10 +
            ranking["rank_Q2F1"] * 0.04 +
            ranking["rank_Q2F2"] * 0.03 +
            ranking["rank_Q2F3"] * 0.02 +
            ranking["rank_n_features"] * 0.01
    )
    ranking = ranking.sort_values("preliminary_rank_score", ascending=True)
    ranking["preliminary_model_rank"] = np.arange(1, len(ranking) + 1)
    return ranking


def leverage_values(X_train, X_apply):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_apply_scaled = scaler.transform(X_apply)
    X_train_design = np.column_stack([np.ones(X_train_scaled.shape[0]), X_train_scaled])
    X_apply_design = np.column_stack([np.ones(X_apply_scaled.shape[0]), X_apply_scaled])
    xtx_inv = np.linalg.pinv(X_train_design.T @ X_train_design)
    h = np.sum((X_apply_design @ xtx_inv) * X_apply_design, axis=1)
    return h


def compute_applicability_domain_for_model(model_key):
    features = model_feature_sets[model_key]
    pred_df = all_predictions[all_predictions["model_key"] == model_key].copy()
    train_pred_df = pred_df[pred_df["Split"] == "train"].copy()
    test_pred_df = pred_df[pred_df["Split"] == "external_test"].copy()
    X_tr = X_train_final_base[features]
    X_te = X_test_final_base[features]
    h_train = leverage_values(X_tr, X_tr)
    h_test = leverage_values(X_tr, X_te)
    p = len(features)
    n = X_tr.shape[0]
    h_star = 3 * (p + 1) / n
    residual_std = np.std(train_pred_df["Residual"].values, ddof=1)
    if residual_std == 0:
        residual_std = np.nan
    train_std_res = train_pred_df["Residual"].values / residual_std
    test_std_res = test_pred_df["Residual"].values / residual_std
    ad_train = pd.DataFrame({
        "model_key": model_key,
        "Molecule_ID": train_pred_df["Molecule_ID"].values,
        "Molecule_Type": train_pred_df["Molecule_Type"].values,
        "Structure_Name": train_pred_df["Structure_Name"].values,
        "Split": "train",
        "Experimental": train_pred_df["Experimental"].values,
        "Predicted": train_pred_df["Predicted"].values,
        "Residual": train_pred_df["Residual"].values,
        "Standardized_Residual": train_std_res,
        "Leverage_h": h_train,
        "h_star": h_star,
        "outside_leverage": h_train > h_star,
        "outside_residual": np.abs(train_std_res) > 3,
        "outside_AD": (h_train > h_star) | (np.abs(train_std_res) > 3)
    })
    ad_test = pd.DataFrame({
        "model_key": model_key,
        "Molecule_ID": test_pred_df["Molecule_ID"].values,
        "Molecule_Type": test_pred_df["Molecule_Type"].values,
        "Structure_Name": test_pred_df["Structure_Name"].values,
        "Split": "external_test",
        "Experimental": test_pred_df["Experimental"].values,
        "Predicted": test_pred_df["Predicted"].values,
        "Residual": test_pred_df["Residual"].values,
        "Standardized_Residual": test_std_res,
        "Leverage_h": h_test,
        "h_star": h_star,
        "outside_leverage": h_test > h_star,
        "outside_residual": np.abs(test_std_res) > 3,
        "outside_AD": (h_test > h_star) | (np.abs(test_std_res) > 3)
    })
    return pd.concat([ad_train, ad_test], axis=0, ignore_index=True)


def y_randomization_test(estimator, X, y, meta, n_randomization=300, real_q2=None):
    rng = np.random.default_rng(RANDOM_STATE)
    records = []
    for i in range(n_randomization):
        y_perm = pd.Series(rng.permutation(y.values), index=y.index)
        try:
            cv_summary, _ = evaluate_cv_model(
                estimator, X, y_perm, meta,
                n_splits=N_SPLITS_CV,
                n_repeats=max(3, min(5, N_REPEATS_CV)),
                random_state=RANDOM_STATE + i + 2000
            )
            records.append({
                "iteration": i + 1,
                "randomized_Q2_mean": cv_summary["Q2_mean"],
                "randomized_RMSE_CV_mean": cv_summary["RMSE_CV_mean"]
            })
        except Exception as e:
            records.append(
                {"iteration": i + 1, "randomized_Q2_mean": np.nan, "randomized_RMSE_CV_mean": np.nan, "error": str(e)})
    rand_df = pd.DataFrame(records)
    if real_q2 is not None and not pd.isna(real_q2):
        valid = rand_df["randomized_Q2_mean"].dropna()
        p_value = float((np.sum(valid >= real_q2) + 1) / (len(valid) + 1)) if len(valid) > 0 else np.nan
    else:
        p_value = np.nan
    return rand_df, p_value


preliminary_ranking = preliminary_rank_models(all_model_metrics)
robust_candidate_keys = preliminary_ranking.head(min(MAX_MODELS_FOR_ROBUST_RANKING, len(preliminary_ranking)))[
    "model_key"].tolist()
print("Robust ranking candidates:")
for key in robust_candidate_keys:
    print("-", key)

applicability_records = []
y_randomization_records = []

for model_key in robust_candidate_keys:
    print(f"Applicability domain and Y-randomization for ranking candidate: {model_key}")
    ad_df = compute_applicability_domain_for_model(model_key)
    applicability_records.append(ad_df)
    model = trained_models[model_key]
    features = model_feature_sets[model_key]
    X_tr = X_train_final_base[features]
    real_q2 = preliminary_ranking.loc[preliminary_ranking["model_key"] == model_key, "Q2_mean"].iloc[0]
    rand_df, p_value = y_randomization_test(model, X_tr, y_train, train_meta, n_randomization=N_Y_RANDOMIZATION,
                                            real_q2=real_q2)
    rand_df["model_key"] = model_key
    rand_df["real_Q2_mean"] = real_q2
    rand_df["y_randomization_p_value"] = p_value
    y_randomization_records.append(rand_df)

applicability_domain = pd.concat(applicability_records, axis=0, ignore_index=True)
y_randomization = pd.concat(y_randomization_records, axis=0, ignore_index=True)
TABLES["applicability_domain"] = applicability_domain
TABLES["y_randomization"] = y_randomization
save_dataframe(applicability_domain, "applicability_domain.xlsx")
save_dataframe(y_randomization, "y_randomization.xlsx")

ad_summary = applicability_domain.groupby("model_key").agg(
    n_total_AD=("Molecule_ID", "count"),
    n_outside_AD=("outside_AD", "sum"),
    n_outside_leverage=("outside_leverage", "sum"),
    n_outside_residual=("outside_residual", "sum")
).reset_index()
ad_summary["outside_AD_fraction"] = ad_summary["n_outside_AD"] / ad_summary["n_total_AD"]

yrand_summary = y_randomization.groupby("model_key").agg(
    real_Q2_mean=("real_Q2_mean", "first"),
    randomized_Q2_mean=("randomized_Q2_mean", "mean"),
    randomized_Q2_std=("randomized_Q2_mean", "std"),
    y_randomization_p_value=("y_randomization_p_value", "first")
).reset_index()

final_ranking = preliminary_ranking[preliminary_ranking["model_key"].isin(robust_candidate_keys)].copy()
final_ranking = final_ranking.merge(ad_summary, on="model_key", how="left")
final_ranking = final_ranking.merge(yrand_summary, on="model_key", how="left")
final_ranking = add_rank_column(final_ranking, "y_randomization_p_value", "rank_y_randomization_p", ascending=True)
final_ranking = add_rank_column(final_ranking, "outside_AD_fraction", "rank_AD", ascending=True)
final_ranking["final_rank_score"] = (
        final_ranking["preliminary_rank_score"] * 0.70 +
        final_ranking["rank_y_randomization_p"] * 0.15 +
        final_ranking["rank_AD"] * 0.15
)
final_ranking = final_ranking.sort_values("final_rank_score", ascending=True)
final_ranking["final_model_rank"] = np.arange(1, len(final_ranking) + 1)

TOP_MODEL_KEYS = final_ranking.head(TOP_N_MODELS)["model_key"].tolist()
top3_metrics = final_ranking.head(TOP_N_MODELS).copy()
TABLES["model_ranking"] = final_ranking
TABLES["top3_model_metrics"] = top3_metrics
save_dataframe(preliminary_ranking, "preliminary_model_ranking.xlsx")
save_dataframe(final_ranking, "model_ranking.xlsx")
save_dataframe(top3_metrics, "top3_model_metrics.xlsx")

fig, ax = plt.subplots(figsize=(10, max(5, 0.35 * len(final_ranking))))
plot_df = final_ranking.sort_values("final_rank_score", ascending=False)
ax.barh(plot_df["model_key"], plot_df["final_rank_score"])
ax.set_xlabel("Final rank score, lower is better")
ax.set_title("Final model ranking with AD and Y-randomization")
save_figure(fig, "model_ranking_final_with_AD_Yrandomization")

for model_key in robust_candidate_keys:
    tmp = y_randomization[y_randomization["model_key"] == model_key]
    real_q2 = tmp["real_Q2_mean"].iloc[0]
    p_value = tmp["y_randomization_p_value"].iloc[0]
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.histplot(tmp["randomized_Q2_mean"].dropna(), kde=True, ax=ax)
    ax.axvline(real_q2, linestyle="--", label=f"Real Q2 = {real_q2:.3f}")
    ax.set_xlabel("Randomized Q2 mean")
    ax.set_title(f"Y-randomization\n{model_key}\np = {p_value:.4f}")
    ax.legend()
    save_figure(fig, f"y_randomization_{model_key}")

print("Final TOP 3 models:")
for key in TOP_MODEL_KEYS:
    print("-", key)
display(top3_metrics)


Robust ranking candidates:
- SVR__model_specific_best_subset__rank_top_10
- GaussianProcess__global_robust_subset__rank_top_8
- SVR__global_robust_subset__rank_top_8
- CatBoost__global_robust_subset__rank_top_8
- GradientBoosting__global_robust_subset__rank_top_8
- XGBoost__global_robust_subset__rank_top_8
- Ridge__model_specific_best_subset__rank_top_20
- ExtraTrees__global_robust_subset__rank_top_8
- ElasticNet__global_robust_subset__rank_top_8
- AdaBoost__global_robust_subset__rank_top_8
Applicability domain and Y-randomization for ranking candidate: SVR__model_specific_best_subset__rank_top_10
Applicability domain and Y-randomization for ranking candidate: GaussianProcess__global_robust_subset__rank_top_8
Applicability domain and Y-randomization for ranking candidate: SVR__global_robust_subset__rank_top_8
Applicability domain and Y-randomization for ranking candidate: CatBoost__global_robust_subset__rank_top_8
Applicability domain and Y-randomization for ranking candidate: Gradient

,model_key,model_name,feature_mode,subset_name,n_features,features,best_params_json,model_path,Q2_mean,Q2_std,...,n_outside_residual,outside_AD_fraction,real_Q2_mean,randomized_Q2_mean,randomized_Q2_std,y_randomization_p_value,rank_y_randomization_p,rank_AD,final_rank_score,final_model_rank
0,SVR__model_specific_best_subset__rank_top_10,SVR,model_specific_best_subset,rank_top_10,10,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,"{""C"": 7.63561552176222, ""epsilon"": 0.000279821...",results_QSPR_QSAR_LogS1_20260702_13\models\SVR...,0.77,0.15,...,3,0.03,0.77,-0.44,0.18,0.00,5.50,2.50,2.34,1
1,GaussianProcess__global_robust_subset__rank_top_8,GaussianProcess,global_robust_subset,rank_top_8,8,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,"{""alpha"": 2.3144513393908483e-07, ""length_scal...",results_QSPR_QSAR_LogS1_20260702_13\models\Gau...,0.71,0.17,...,5,0.06,0.71,-0.18,0.10,0.00,5.50,4.00,3.50,2
2,SVR__global_robust_subset__rank_top_8,SVR,global_robust_subset,rank_top_8,8,Koopmans_EA; Se1Cu2Ag1s:(OEstate); Dipole_mome...,"{""C"": 609.1394159031862, ""epsilon"": 0.00131839...",results_QSPR_QSAR_LogS1_20260702_13\models\SVR...,0.75,0.17,...,2,0.03,0.75,-0.32,0.14,0.00,5.50,2.50,3.53,3


## 13. Prediction diagnostics for the top three models

Diagnostic plots are generated for each selected model using training and external-test predictions.

The output includes:

- predicted versus experimental values;
- residual plots;
- Williams plots for applicability-domain assessment.

Williams plots display leverage against standardized residuals and are used to identify observations outside the descriptor-space or residual-based applicability-domain limits. Figures are saved in both PNG and PDF formats.


In [14]:
# ============================================================
# Prediction diagnostics and Williams plots
# ============================================================

top3_predictions = all_predictions[all_predictions["model_key"].isin(TOP_MODEL_KEYS)].copy()
TABLES["top3_predictions"] = top3_predictions
save_dataframe(top3_predictions, "top3_predictions.xlsx")


def plot_predicted_vs_experimental(pred_df, model_key):
    fig, ax = plt.subplots(figsize=(6, 6))
    for split_name, tmp in pred_df.groupby("Split"):
        ax.scatter(tmp["Experimental"], tmp["Predicted"], label=split_name, alpha=0.85)
    min_val = min(pred_df["Experimental"].min(), pred_df["Predicted"].min())
    max_val = max(pred_df["Experimental"].max(), pred_df["Predicted"].max())
    ax.plot([min_val, max_val], [min_val, max_val], linestyle="--", linewidth=1.5)
    ax.set_xlabel("Experimental Log_S1")
    ax.set_ylabel("Predicted Log_S1")
    ax.set_title(f"Predicted vs Experimental\n{model_key}")
    ax.legend()
    save_figure(fig, f"predicted_vs_experimental_{model_key}")


def plot_residuals(pred_df, model_key):
    fig, ax = plt.subplots(figsize=(7, 5))
    for split_name, tmp in pred_df.groupby("Split"):
        ax.scatter(tmp["Predicted"], tmp["Residual"], label=split_name, alpha=0.85)
    ax.axhline(0, linestyle="--", linewidth=1.5)
    ax.set_xlabel("Predicted Log_S1")
    ax.set_ylabel("Residual = Experimental - Predicted")
    ax.set_title(f"Residual plot\n{model_key}")
    ax.legend()
    save_figure(fig, f"residual_plot_{model_key}")


def plot_williams(ad_df, model_key):
    fig, ax = plt.subplots(figsize=(7, 5))
    for split_name, tmp in ad_df.groupby("Split"):
        ax.scatter(tmp["Leverage_h"], tmp["Standardized_Residual"], label=split_name, alpha=0.85)
    h_star = ad_df["h_star"].iloc[0]
    ax.axvline(h_star, linestyle="--", label=f"h* = {h_star:.3f}")
    ax.axhline(3, linestyle="--")
    ax.axhline(-3, linestyle="--")
    ax.set_xlabel("Leverage h")
    ax.set_ylabel("Standardized residual")
    ax.set_title(f"Williams plot\n{model_key}")
    ax.legend()
    save_figure(fig, f"williams_plot_{model_key}")


for model_key in TOP_MODEL_KEYS:
    pred_df = top3_predictions[top3_predictions["model_key"] == model_key].copy()
    plot_predicted_vs_experimental(pred_df, model_key)
    plot_residuals(pred_df, model_key)
    ad_df = applicability_domain[applicability_domain["model_key"] == model_key].copy()
    if not ad_df.empty:
        plot_williams(ad_df, model_key)

outside_ad_top3 = applicability_domain[
    (applicability_domain["model_key"].isin(TOP_MODEL_KEYS)) & (applicability_domain["outside_AD"])]
display(outside_ad_top3.sort_values(["model_key", "Split", "Molecule_ID"]))


,model_key,Molecule_ID,Molecule_Type,Structure_Name,Split,Experimental,Predicted,Residual,Standardized_Residual,Leverage_h,h_star,outside_leverage,outside_residual,outside_AD
182,GaussianProcess__global_robust_subset__rank_top_8,18,complex,complex_18,external_test,2.66,2.62,0.04,3.72,0.10,0.34,False,True,True
183,GaussianProcess__global_robust_subset__rank_top_8,20,complex,complex_20,external_test,2.86,2.82,0.04,3.84,0.15,0.34,False,True,True
188,GaussianProcess__global_robust_subset__rank_top_8,44,complex,complex_44,external_test,2.64,2.59,0.05,4.32,0.09,0.34,False,True,True
198,GaussianProcess__global_robust_subset__rank_top_8,92,complex,complex_92,external_test,2.56,2.51,0.05,4.27,0.11,0.34,False,True,True
100,GaussianProcess__global_robust_subset__rank_top_8,1,bimetal,bimetal_1,train,2.59,2.59,-0.00,-0.04,0.36,0.34,True,False,True
113,GaussianProcess__global_robust_subset__rank_top_8,16,complex,complex_16,train,2.68,2.65,0.04,3.32,0.10,0.34,False,True,True
283,SVR__global_robust_subset__rank_top_8,20,complex,complex_20,external_test,2.86,2.78,0.08,4.73,0.15,0.34,False,True,True
200,SVR__global_robust_subset__rank_top_8,1,bimetal,bimetal_1,train,2.59,2.59,-0.00,-0.09,0.36,0.34,True,False,True
213,SVR__global_robust_subset__rank_top_8,16,complex,complex_16,train,2.68,2.62,0.06,3.52,0.10,0.34,False,True,True
83,SVR__model_specific_best_subset__rank_top_10,20,complex,complex_20,external_test,2.86,2.78,0.08,4.99,0.15,0.41,False,True,True


## 14. Robustness validation of the top three models

Additional post-selection validation is performed for the three final models.

Monte Carlo complex-only validation repeatedly generates random validation subsets composed of complexes while retaining bimetallic clusters and individual bases in the training portion. Bootstrap validation evaluates the stability of external-test metrics under resampling of the training set. Nested cross-validation repeats hyperparameter optimization within each outer fold to assess sensitivity to model tuning.

The resulting Q², RMSE, MAE, R², and CCC distributions are stored for comparison of model stability.


In [15]:
# ============================================================
# Monte Carlo, bootstrap, and nested cross-validation
# ============================================================


def monte_carlo_complex_validation(estimator, X, y, meta, n_splits=300, test_fraction=0.25):
    records = []
    meta_reset = meta.reset_index(drop=True)
    complex_positions = np.where(meta_reset["Molecule_Type"].values == "complex")[0]
    fixed_train_positions = np.where(meta_reset["Molecule_Type"].values != "complex")[0]
    rng = np.random.default_rng(RANDOM_STATE)
    for i in range(n_splits):
        n_val = max(2, int(round(len(complex_positions) * test_fraction)))
        n_val = min(n_val, len(complex_positions) - 1)
        val_positions = rng.choice(complex_positions, size=n_val, replace=False)
        train_complex_positions = np.array([x for x in complex_positions if x not in set(val_positions)])
        train_positions = np.sort(np.concatenate([fixed_train_positions, train_complex_positions]))
        val_positions = np.sort(val_positions)
        model = clone(estimator)
        try:
            model.fit(X.iloc[train_positions], y.iloc[train_positions])
            pred_val = predict_1d(model, X.iloc[val_positions])
            records.append({
                "iteration": i + 1,
                "Q2": r2_score(y.iloc[val_positions], pred_val) if len(val_positions) > 1 else np.nan,
                "RMSE": rmse(y.iloc[val_positions], pred_val),
                "MAE": mean_absolute_error(y.iloc[val_positions], pred_val),
                "n_train": len(train_positions),
                "n_validation": len(val_positions)
            })
        except Exception as e:
            records.append(
                {"iteration": i + 1, "Q2": np.nan, "RMSE": np.nan, "MAE": np.nan, "n_train": len(train_positions),
                 "n_validation": len(val_positions), "error": str(e)})
    return pd.DataFrame(records)


def bootstrap_external_validation(estimator, X_train, y_train_local, X_test, y_test_local, n_bootstrap=500):
    rng = np.random.default_rng(RANDOM_STATE)
    records = []
    n = len(y_train_local)
    for i in range(n_bootstrap):
        boot_idx = rng.choice(np.arange(n), size=n, replace=True)
        oob_idx = np.array(sorted(list(set(np.arange(n)) - set(boot_idx))))
        model = clone(estimator)
        try:
            model.fit(X_train.iloc[boot_idx], y_train_local.iloc[boot_idx])
            pred_test = predict_1d(model, X_test)
            record = {
                "iteration": i + 1,
                "R2_test": r2_score(y_test_local, pred_test) if len(y_test_local) > 1 else np.nan,
                "RMSE_test": rmse(y_test_local, pred_test),
                "MAE_test": mean_absolute_error(y_test_local, pred_test),
                "CCC_test": concordance_correlation_coefficient(y_test_local, pred_test),
                "n_bootstrap_train": len(boot_idx),
                "n_oob": len(oob_idx)
            }
            if len(oob_idx) >= 2:
                pred_oob = predict_1d(model, X_train.iloc[oob_idx])
                record["R2_oob"] = r2_score(y_train_local.iloc[oob_idx], pred_oob)
                record["RMSE_oob"] = rmse(y_train_local.iloc[oob_idx], pred_oob)
            else:
                record["R2_oob"] = np.nan
                record["RMSE_oob"] = np.nan
            records.append(record)
        except Exception as e:
            records.append(
                {"iteration": i + 1, "R2_test": np.nan, "RMSE_test": np.nan, "MAE_test": np.nan, "CCC_test": np.nan,
                 "R2_oob": np.nan, "RMSE_oob": np.nan, "error": str(e)})
    return pd.DataFrame(records)


def nested_cv_for_model(model_name, X, y, meta):
    outer_splits = make_complex_only_cv_splits(meta, n_splits=N_SPLITS_CV, n_repeats=N_NESTED_OUTER_REPEATS,
                                               random_state=RANDOM_STATE + 1000)
    records = []
    for outer_id, (outer_train_idx, outer_val_idx) in enumerate(outer_splits, start=1):
        X_outer_train = X.iloc[outer_train_idx].reset_index(drop=True)
        y_outer_train = y.iloc[outer_train_idx].reset_index(drop=True)
        meta_outer_train = meta.iloc[outer_train_idx].reset_index(drop=True)
        X_outer_val = X.iloc[outer_val_idx]
        y_outer_val = y.iloc[outer_val_idx]
        try:
            estimator, params, _ = optimize_model_with_optuna(model_name, X_outer_train, y_outer_train,
                                                              meta_outer_train, n_trials=N_NESTED_OPTUNA_TRIALS)
            estimator.fit(X_outer_train, y_outer_train)
            pred_outer = predict_1d(estimator, X_outer_val)
            records.append({
                "outer_fold": outer_id,
                "model_name": model_name,
                "Q2_outer": r2_score(y_outer_val, pred_outer) if len(y_outer_val) > 1 else np.nan,
                "RMSE_outer": rmse(y_outer_val, pred_outer),
                "MAE_outer": mean_absolute_error(y_outer_val, pred_outer),
                "best_params_json": json.dumps(params, ensure_ascii=False)
            })
        except Exception as e:
            records.append({"outer_fold": outer_id, "model_name": model_name, "Q2_outer": np.nan, "RMSE_outer": np.nan,
                            "MAE_outer": np.nan, "error": str(e)})
    return pd.DataFrame(records)


monte_carlo_records = []
bootstrap_records = []
nested_records = []
for model_key in TOP_MODEL_KEYS:
    model_name = top3_metrics.loc[top3_metrics["model_key"] == model_key, "model_name"].iloc[0]
    features = model_feature_sets[model_key]
    estimator = trained_models[model_key]
    X_tr = X_train_final_base[features]
    X_te = X_test_final_base[features]
    print(f"Robustness validation for {model_key}")
    mc_df = monte_carlo_complex_validation(estimator, X_tr, y_train, train_meta, n_splits=N_MONTE_CARLO_SPLITS)
    mc_df["model_key"] = model_key
    monte_carlo_records.append(mc_df)
    boot_df = bootstrap_external_validation(estimator, X_tr, y_train, X_te, y_test, n_bootstrap=N_BOOTSTRAP)
    boot_df["model_key"] = model_key
    bootstrap_records.append(boot_df)
    nested_df = nested_cv_for_model(model_name, X_tr, y_train, train_meta)
    nested_df["model_key"] = model_key
    nested_records.append(nested_df)

monte_carlo_validation = pd.concat(monte_carlo_records, axis=0, ignore_index=True)
bootstrap_validation = pd.concat(bootstrap_records, axis=0, ignore_index=True)
nested_cv_validation = pd.concat(nested_records, axis=0, ignore_index=True)
TABLES["monte_carlo_validation"] = monte_carlo_validation
TABLES["bootstrap_validation"] = bootstrap_validation
TABLES["nested_cv_validation"] = nested_cv_validation
save_dataframe(monte_carlo_validation, "monte_carlo_validation.xlsx")
save_dataframe(bootstrap_validation, "bootstrap_validation.xlsx")
save_dataframe(nested_cv_validation, "nested_cv_validation.xlsx")

for model_key in TOP_MODEL_KEYS:
    mc_tmp = monte_carlo_validation[monte_carlo_validation["model_key"] == model_key]
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.histplot(mc_tmp["Q2"].dropna(), kde=True, ax=ax)
    ax.set_xlabel("Monte Carlo Q2")
    ax.set_title(f"Monte Carlo validation Q2\n{model_key}")
    save_figure(fig, f"monte_carlo_q2_{model_key}")
    boot_tmp = bootstrap_validation[bootstrap_validation["model_key"] == model_key]
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.histplot(boot_tmp["RMSE_test"].dropna(), kde=True, ax=ax)
    ax.set_xlabel("Bootstrap external RMSE")
    ax.set_title(f"Bootstrap external RMSE\n{model_key}")
    save_figure(fig, f"bootstrap_rmse_test_{model_key}")

display(monte_carlo_validation.groupby("model_key")[["Q2", "RMSE", "MAE"]].agg(["mean", "std"]))
display(
    bootstrap_validation.groupby("model_key")[["R2_test", "RMSE_test", "MAE_test", "CCC_test"]].agg(["mean", "std"]))


Robustness validation for SVR__model_specific_best_subset__rank_top_10
Robustness validation for GaussianProcess__global_robust_subset__rank_top_8
Robustness validation for SVR__global_robust_subset__rank_top_8


Q2      RMSE       MAE  \
                                                  mean  std mean  std mean   
model_key                                                                    
GaussianProcess__global_robust_subset__rank_top_8 0.70 0.18 0.03 0.01 0.02   
SVR__global_robust_subset__rank_top_8             0.75 0.16 0.03 0.01 0.02   
SVR__model_specific_best_subset__rank_top_10      0.75 0.17 0.03 0.01 0.02   

                                                        
                                                   std  
model_key                                               
GaussianProcess__global_robust_subset__rank_top_8 0.01  
SVR__global_robust_subset__rank_top_8             0.00  
SVR__model_specific_best_subset__rank_top_10      0.00

R2_test      RMSE_test       \
                                                     mean  std      mean  std   
model_key                                                                       
GaussianProcess__global_robust_subset__rank_top_8    0.59 0.19      0.06 0.01   
SVR__global_robust_subset__rank_top_8                0.85 0.06      0.04 0.01   
SVR__model_specific_best_subset__rank_top_10         0.86 0.06      0.03 0.01   

                                                  MAE_test      CCC_test       
                                                      mean  std     mean  std  
model_key                                                                      
GaussianProcess__global_robust_subset__rank_top_8     0.04 0.01     0.71 0.16  
SVR__global_robust_subset__rank_top_8                 0.03 0.00     0.91 0.05  
SVR__model_specific_best_subset__rank_top_10          0.02 0.00     0.92 0.04

## 15. Model interpretation

The final models are interpreted using complementary feature-importance and effect-estimation methods.

For each model, permutation importance is calculated and SHAP values are estimated when supported. SHAP summary, bar, and dependence plots are generated for the leading descriptors. One-dimensional accumulated local effects (ALE) are calculated for the most important features, and Spearman correlations with `Log_S1` are reported alongside the ALE trend.

The combined results are summarized in `permutation_importance_top3`, `shap_importance_top3`, and `descriptor_direction_summary`.


In [16]:
# ============================================================
# Interpretation of the selected models
# ============================================================


def compute_model_permutation_importance(model, X, y, model_key):
    perm = permutation_importance(model, X, y, n_repeats=N_PERMUTATION_REPEATS, random_state=RANDOM_STATE, n_jobs=-1,
                                  scoring="neg_root_mean_squared_error")
    return pd.DataFrame({
        "model_key": model_key,
        "feature": X.columns,
        "permutation_importance_mean": perm.importances_mean,
        "permutation_importance_std": perm.importances_std
    }).sort_values("permutation_importance_mean", ascending=False)


def manual_ale_1d(model, X, feature, bins=10):
    x = X[feature].values.astype(float)
    quantiles = np.unique(np.quantile(x, np.linspace(0, 1, bins + 1)))
    if len(quantiles) < 3:
        return pd.DataFrame(columns=["feature_value", "ale"])
    bin_effects, bin_centers = [], []
    for low, high in zip(quantiles[:-1], quantiles[1:]):
        if low == high:
            continue
        mask = (x >= low) & (x <= high)
        if not np.any(mask):
            bin_effects.append(0.0)
            bin_centers.append((low + high) / 2)
            continue
        X_low = X.loc[mask].copy()
        X_high = X.loc[mask].copy()
        X_low[feature] = low
        X_high[feature] = high
        pred_low = predict_1d(model, X_low)
        pred_high = predict_1d(model, X_high)
        bin_effects.append(float(np.mean(pred_high - pred_low)))
        bin_centers.append((low + high) / 2)
    ale_values = np.cumsum(bin_effects)
    ale_values = ale_values - np.mean(ale_values)
    return pd.DataFrame({"feature_value": bin_centers, "ale": ale_values})


interpretation_perm_records = []
interpretation_shap_records = []
direction_records = []

for model_key in TOP_MODEL_KEYS:
    model = trained_models[model_key]
    features = model_feature_sets[model_key]
    X_tr = X_train_final_base[features].copy()
    print(f"Interpretation for {model_key}")

    perm_df = compute_model_permutation_importance(model, X_tr, y_train, model_key)
    interpretation_perm_records.append(perm_df)

    fig, ax = plt.subplots(figsize=(8, max(5, 0.3 * min(25, len(perm_df)))))
    plot_perm = perm_df.head(25).sort_values("permutation_importance_mean", ascending=True)
    ax.barh(plot_perm["feature"], plot_perm["permutation_importance_mean"])
    ax.set_xlabel("Permutation importance, RMSE increase")
    ax.set_ylabel("Feature")
    ax.set_title(f"Permutation importance\n{model_key}")
    save_figure(fig, f"permutation_importance_{model_key}")

    top_features = perm_df.head(min(N_TOP_FEATURES_FOR_INTERPRETATION, len(perm_df)))["feature"].tolist()

    shap_summary_available = False
    if HAS_SHAP:
        try:
            X_shap = X_tr.copy()
            if len(X_shap) > N_SHAP_MAX_SAMPLES:
                X_shap = X_shap.sample(N_SHAP_MAX_SAMPLES, random_state=RANDOM_STATE)
            if isinstance(model, Pipeline):
                final_model = model.named_steps["model"]
                X_for_shap = pd.DataFrame(model.named_steps["scaler"].transform(X_shap), columns=X_shap.columns,
                                          index=X_shap.index)
            else:
                final_model = model
                X_for_shap = X_shap
            if hasattr(final_model, "feature_importances_"):
                explainer = shap.TreeExplainer(final_model)
                shap_values = explainer.shap_values(X_for_shap)
            else:
                background = shap.kmeans(X_for_shap, min(10, len(X_for_shap)))
                explainer = shap.KernelExplainer(final_model.predict, background)
                shap_values = explainer.shap_values(X_for_shap, nsamples=min(100, 2 * X_for_shap.shape[1] + 1))
            shap_values = np.asarray(shap_values)
            if shap_values.ndim == 3:
                shap_values = shap_values[:, :, 0]
            mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
            shap_df = pd.DataFrame(
                {"model_key": model_key, "feature": X_shap.columns, "mean_abs_shap": mean_abs_shap}).sort_values(
                "mean_abs_shap", ascending=False)
            interpretation_shap_records.append(shap_df)

            plt.figure(figsize=(8, 6))
            shap.summary_plot(shap_values, X_for_shap, show=False, max_display=20)
            save_figure(plt.gcf(), f"shap_summary_{model_key}")
            plt.figure(figsize=(8, 6))
            shap.summary_plot(shap_values, X_for_shap, plot_type="bar", show=False, max_display=20)
            save_figure(plt.gcf(), f"shap_bar_{model_key}")
            for feature in shap_df.head(min(3, len(shap_df)))["feature"].tolist():
                plt.figure(figsize=(7, 5))
                shap.dependence_plot(feature, shap_values, X_for_shap, show=False)
                save_figure(plt.gcf(), f"shap_dependence_{model_key}_{feature}")
            shap_summary_available = True
        except Exception as e:
            add_warning(f"SHAP failed for {model_key}: {e}")
    if not shap_summary_available:
        interpretation_shap_records.append(
            pd.DataFrame({"model_key": [model_key], "feature": ["SHAP not available"], "mean_abs_shap": [np.nan]}))

    for feature in top_features:
        ale_df = manual_ale_1d(model, X_tr, feature, bins=N_ALE_BINS)
        if not ale_df.empty:
            fig, ax = plt.subplots(figsize=(7, 5))
            ax.plot(ale_df["feature_value"], ale_df["ale"], marker="o")
            ax.set_xlabel(feature)
            ax.set_ylabel("ALE")
            ax.set_title(f"ALE plot: {feature}\n{model_key}")
            save_figure(fig, f"ale_{model_key}_{feature}")
            ale_trend = ale_df["ale"].iloc[-1] - ale_df["ale"].iloc[0]
            ale_direction = "positive" if ale_trend > 0 else "negative" if ale_trend < 0 else "flat"
        else:
            ale_trend = np.nan
            ale_direction = "not enough unique values"
        spearman_rho, spearman_p = spearmanr(X_tr[feature], y_train)
        direction_records.append({
            "model_key": model_key,
            "feature": feature,
            "spearman_rho_with_Log_S1": spearman_rho,
            "spearman_p_value": spearman_p,
            "ale_trend_last_minus_first": ale_trend,
            "ale_direction": ale_direction
        })

permutation_importance_top3 = pd.concat(interpretation_perm_records, axis=0, ignore_index=True)
shap_importance_top3 = pd.concat(interpretation_shap_records, axis=0, ignore_index=True)
descriptor_direction_summary = pd.DataFrame(direction_records)
TABLES["permutation_importance_top3"] = permutation_importance_top3
TABLES["shap_importance_top3"] = shap_importance_top3
TABLES["descriptor_direction_summary"] = descriptor_direction_summary
save_dataframe(permutation_importance_top3, "permutation_importance_top3.xlsx")
save_dataframe(shap_importance_top3, "shap_importance_top3.xlsx")
save_dataframe(descriptor_direction_summary, "descriptor_direction_summary.xlsx")

display(permutation_importance_top3.groupby("model_key").head(10))
display(descriptor_direction_summary)


Interpretation for SVR__model_specific_best_subset__rank_top_10


  0%|          | 0/80 [00:00<?, ?it/s]

Interpretation for GaussianProcess__global_robust_subset__rank_top_8


  0%|          | 0/80 [00:00<?, ?it/s]

Interpretation for SVR__global_robust_subset__rank_top_8


...

<Figure size 840x600 with 0 Axes>

<Figure size 840x600 with 0 Axes>

<Figure size 840x600 with 0 Axes>

<Figure size 840x600 with 0 Axes>

<Figure size 840x600 with 0 Axes>

## 16. Ensemble predictions

Simple and weighted-average ensembles are constructed from the predictions of the three selected models.

The simple ensemble assigns equal weight to all models. The weighted ensemble uses inverse repeated-CV RMSE as the weighting criterion. External-test R², RMSE, MAE, CCC, Q²F1–Q²F3, and Golbraikh–Tropsha statistics are calculated for both ensemble variants.

Ensemble results are retained as an additional comparison and do not replace the individually selected models by default.


In [17]:
# ============================================================
# Ensemble predictions
# ============================================================

ensemble_df = test_meta[["Molecule_ID", "Molecule_Type", "Structure_Name"]].copy()
ensemble_df["Experimental"] = y_test.values

for model_key in TOP_MODEL_KEYS:
    pred_test = \
        top3_predictions[(top3_predictions["model_key"] == model_key) & (top3_predictions["Split"] == "external_test")][
            ["Molecule_ID", "Predicted"]].rename(columns={"Predicted": model_key})
    ensemble_df = ensemble_df.merge(pred_test, on="Molecule_ID", how="left")

prediction_cols = TOP_MODEL_KEYS
ensemble_df["simple_average_prediction"] = ensemble_df[prediction_cols].mean(axis=1)
weights_df = top3_metrics.set_index("model_key").loc[TOP_MODEL_KEYS]
rmse_values = weights_df["RMSE_CV_mean"].replace(0, np.nan)
raw_weights = 1 / rmse_values
raw_weights = raw_weights.replace([np.inf, -np.inf], np.nan)
if raw_weights.isna().all() or raw_weights.sum(skipna=True) == 0:
    weights = pd.Series(np.ones(len(TOP_MODEL_KEYS)) / len(TOP_MODEL_KEYS), index=TOP_MODEL_KEYS)
else:
    raw_weights = raw_weights.fillna(raw_weights.dropna().mean())
    weights = raw_weights / raw_weights.sum()

ensemble_df["weighted_average_prediction"] = 0.0
for model_key in TOP_MODEL_KEYS:
    ensemble_df["weighted_average_prediction"] += ensemble_df[model_key] * weights.loc[model_key]

ensemble_metrics_records = []
for pred_col in ["simple_average_prediction", "weighted_average_prediction"]:
    y_pred = ensemble_df[pred_col].values
    record = {
        "ensemble_type": pred_col,
        "R2_test": r2_score(y_test, y_pred),
        "RMSE_test": rmse(y_test, y_pred),
        "MAE_test": mean_absolute_error(y_test, y_pred),
        "CCC": concordance_correlation_coefficient(y_test, y_pred),
        "Q2F1": q2_f1(y_train, y_test, y_pred),
        "Q2F2": q2_f2(y_test, y_pred),
        "Q2F3": q2_f3(y_train, y_test, y_pred),
        "weights_json": json.dumps(weights.to_dict(), ensure_ascii=False)
    }
    record.update(golbraikh_tropsha_metrics(y_test, y_pred))
    ensemble_metrics_records.append(record)

ensemble_metrics = pd.DataFrame(ensemble_metrics_records)
TABLES["ensemble_predictions"] = ensemble_df.copy()
TABLES["ensemble_metrics"] = ensemble_metrics
save_dataframe(ensemble_df, "ensemble_predictions.xlsx")
save_dataframe(ensemble_metrics, "ensemble_metrics.xlsx")

for pred_col in ["simple_average_prediction", "weighted_average_prediction"]:
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(ensemble_df["Experimental"], ensemble_df[pred_col], alpha=0.85)
    min_val = min(ensemble_df["Experimental"].min(), ensemble_df[pred_col].min())
    max_val = max(ensemble_df["Experimental"].max(), ensemble_df[pred_col].max())
    ax.plot([min_val, max_val], [min_val, max_val], linestyle="--")
    ax.set_xlabel("Experimental Log_S1")
    ax.set_ylabel("Predicted Log_S1")
    ax.set_title(f"External test ensemble\n{pred_col}")
    save_figure(fig, f"ensemble_predicted_vs_experimental_{pred_col}")

display(ensemble_metrics)
display(ensemble_df)


,ensemble_type,R2_test,RMSE_test,MAE_test,CCC,Q2F1,Q2F2,Q2F3,weights_json,GT_r2,GT_r0_squared,GT_r0prime_squared,GT_k,GT_kprime,GT_abs_r2_minus_r0sq_over_r2,GT_abs_r2_minus_r0psq_over_r2,GT_criteria_pass
0,simple_average_prediction,0.91,0.03,0.02,0.95,0.92,0.91,0.87,"{""SVR__model_specific_best_subset__rank_top_10...",0.91,0.92,0.90,1.00,1.00,0.01,0.02,True
1,weighted_average_prediction,0.91,0.03,0.02,0.95,0.92,0.91,0.87,"{""SVR__model_specific_best_subset__rank_top_10...",0.91,0.92,0.89,1.00,1.00,0.01,0.02,True


,Molecule_ID,Molecule_Type,Structure_Name,Experimental,SVR__model_specific_best_subset__rank_top_10,GaussianProcess__global_robust_subset__rank_top_8,SVR__global_robust_subset__rank_top_8,simple_average_prediction,weighted_average_prediction
0,8,complex,complex_8,2.78,2.77,2.77,2.74,2.76,2.76
1,12,complex,complex_12,2.77,2.76,2.77,2.75,2.76,2.76
2,18,complex,complex_18,2.66,2.63,2.62,2.63,2.63,2.63
3,20,complex,complex_20,2.86,2.78,2.82,2.78,2.79,2.79
4,27,complex,complex_27,2.57,2.56,2.55,2.55,2.55,2.55
5,31,complex,complex_31,2.56,2.56,2.57,2.56,2.56,2.56
6,35,complex,complex_35,2.56,2.58,2.57,2.58,2.58,2.58
7,37,complex,complex_37,2.57,2.58,2.58,2.59,2.58,2.58
8,44,complex,complex_44,2.64,2.60,2.59,2.59,2.60,2.60
9,49,complex,complex_49,2.53,2.53,2.52,2.53,2.53,2.53


## 17. Final feature tables and automatic report

This section consolidates the main outputs of the analysis.

The descriptors used by each of the top three models are collected in `final_selected_features`. Applicability-domain, Y-randomization, bootstrap, and Monte Carlo summaries are merged with the model metrics to form `report_summary`.

An automatic Markdown report, `automatic_qspr_report.md`, is generated from the stored results and run metadata.


In [18]:
# ============================================================
# Final feature tables and report generation
# ============================================================

final_selected_feature_records = []
for model_key in TOP_MODEL_KEYS:
    model_row = top3_metrics[top3_metrics["model_key"] == model_key].iloc[0]
    features = model_feature_sets[model_key]
    for rank_i, feature in enumerate(features, start=1):
        final_selected_feature_records.append({
            "model_key": model_key,
            "final_model_rank": int(model_row["final_model_rank"]),
            "model_name": model_row["model_name"],
            "feature_mode": model_row["feature_mode"],
            "subset_name": model_row["subset_name"],
            "feature_rank_within_subset": rank_i,
            "feature": feature
        })
final_selected_features = pd.DataFrame(final_selected_feature_records)
TABLES["final_selected_features"] = final_selected_features

bootstrap_summary = bootstrap_validation.groupby("model_key").agg(
    bootstrap_R2_test_mean=("R2_test", "mean"),
    bootstrap_R2_test_std=("R2_test", "std"),
    bootstrap_RMSE_test_mean=("RMSE_test", "mean"),
    bootstrap_RMSE_test_std=("RMSE_test", "std"),
    bootstrap_CCC_test_mean=("CCC_test", "mean"),
    bootstrap_CCC_test_std=("CCC_test", "std")
).reset_index()
mc_summary = monte_carlo_validation.groupby("model_key").agg(
    monte_carlo_Q2_mean=("Q2", "mean"),
    monte_carlo_Q2_std=("Q2", "std"),
    monte_carlo_RMSE_mean=("RMSE", "mean"),
    monte_carlo_RMSE_std=("RMSE", "std")
).reset_index()

report_summary = top3_metrics.copy()
report_summary = report_summary.merge(ad_summary, on="model_key", how="left")
report_summary = report_summary.merge(yrand_summary, on="model_key", how="left")
report_summary = report_summary.merge(bootstrap_summary, on="model_key", how="left")
report_summary = report_summary.merge(mc_summary, on="model_key", how="left")
TABLES["report_summary"] = report_summary
save_dataframe(final_selected_features, "final_selected_features.xlsx")
save_dataframe(report_summary, "report_summary.xlsx")


def format_float(x, digits=4):
    try:
        if pd.isna(x):
            return "NA"
        return f"{float(x):.{digits}f}"
    except Exception:
        return str(x)


top3_report_lines = []
for _, row in top3_metrics.iterrows():
    top3_report_lines.append(
        f"- **{int(row['final_model_rank'])}. {row['model_key']}**: "
        f"CV Q2 = {format_float(row['Q2_mean'])} ± {format_float(row['Q2_std'])}, "
        f"CV RMSE = {format_float(row['RMSE_CV_mean'])}, "
        f"external R2 = {format_float(row['R2_test'])}, external RMSE = {format_float(row['RMSE_test'])}, "
        f"CCC = {format_float(row['CCC'])}, features = {int(row['n_features'])}, "
        f"Y-rand p = {format_float(row['y_randomization_p_value'], 5)}, outside AD fraction = {format_float(row['outside_AD_fraction'])}."
    )

outside_ad_lines = []
for _, row in ad_summary[ad_summary["model_key"].isin(TOP_MODEL_KEYS)].iterrows():
    outside_ad_lines.append(
        f"- {row['model_key']}: outside AD = {int(row['n_outside_AD'])}/{int(row['n_total_AD'])}; "
        f"outside leverage = {int(row['n_outside_leverage'])}; outside standardized residual = {int(row['n_outside_residual'])}."
    )

yrand_lines = []
for _, row in yrand_summary[yrand_summary["model_key"].isin(TOP_MODEL_KEYS)].iterrows():
    yrand_lines.append(
        f"- {row['model_key']}: real Q2 = {format_float(row['real_Q2_mean'])}; "
        f"mean randomized Q2 = {format_float(row['randomized_Q2_mean'])}; p-value = {format_float(row['y_randomization_p_value'], 5)}."
    )

interpretation_lines = []
for model_key in TOP_MODEL_KEYS:
    perm_tmp = permutation_importance_top3[permutation_importance_top3["model_key"] == model_key].head(5)
    interpretation_lines.append(
        f"- {model_key}: top permutation descriptors: {', '.join(perm_tmp['feature'].tolist())}.")

missing_descriptor_text = "None" if len(missing_descriptors) == 0 else ", ".join(missing_descriptors)
best_model_key = TOP_MODEL_KEYS[0]

report_text = f"""# Automatic QSPR/QSAR report for Log_S1

Generated: {datetime.now().isoformat()}

## 1. Data description

Dataset size: **{len(df)} molecules**.

Endpoint: **{target_col}**.

External test set Molecule_ID values:

`{EXTERNAL_TEST_IDS}`

Train/internal set size: **{len(train_df_raw)}**  
External test set size: **{len(test_df_raw)}**

## 2. Descriptor list

Descriptor list source: `{DESCRIPTOR_FILE}`

Descriptors listed in TXT: **{len(selected_descriptors_txt)}**  
Descriptors found in Excel: **{len(found_descriptors)}**  
Descriptors missing in Excel: **{len(missing_descriptors)}**

Missing descriptors:

{missing_descriptor_text}

## 3. Manual molecule type assignment

- Molecule_ID 1–6: bimetal
- Molecule_ID 7: base, cytosine
- Molecule_ID 26: base, tyrosine_or_thymine
- Molecule_ID 45: base, adenine
- Molecule_ID 73: base, guanine
- all other molecules: complex

## 4. Train/test split

External test molecules were never used for preprocessing decisions, feature selection, hyperparameter optimization or internal CV.

Internal validation used complex-only validation folds. Bimetals and bases always remained in the training portion.

## 5. Preprocessing

Train-only preprocessing:

- inf replaced by NaN;
- descriptors with >50% NaN on train removed;
- train medians used for imputation;
- constant train descriptors removed.

Descriptors after preprocessing and optional correlation/VIF settings: **{X_train_final_base.shape[1]}**.

## 6. Correlation and VIF diagnostics

Correlation filtering: **{DO_CORRELATION_FILTERING}**; cutoff = **{CORRELATION_CUTOFF}**.  
VIF filtering: **{DO_VIF_FILTERING}**; cutoff = **{VIF_CUTOFF}**.

High-correlation pairs with |r| > {CORRELATION_CUTOFF}: **{len(high_corr_pairs)}**.  
High-VIF features with VIF > {VIF_CUTOFF}: **{len(high_vif)}**.

Correlation and VIF were diagnostic by default because descriptors were already pre-filtered before GA feature selection.

## 7. Target distribution

Train target summary:

- min = {format_float(target_stats['min'])}
- max = {format_float(target_stats['max'])}
- mean = {format_float(target_stats['mean'])}
- std = {format_float(target_stats['std'])}
- skewness = {format_float(target_stats['skewness'])}

Because the endpoint is already Log_S1, no additional log-transform was applied by default.

## 8. Feature subset selection strategy

Candidate subsets included all cleaned GA-selected descriptors, LASSO subset, RFE subset, permutation subset, SHAP subset and aggregated-rank top N subsets.

Selector models used where available:

`{selector_models}`

Global robust subset: **{GLOBAL_ROBUST_SUBSET_NAME}**, n = **{len(GLOBAL_ROBUST_FEATURES)}**.

## 9. TOP 3 model ranking

Final ranking used preliminary model performance plus Y-randomization and applicability-domain diagnostics for robust candidates.

{chr(10).join(top3_report_lines)}

Best ranked model: **{best_model_key}**

## 10. External validation

External validation metrics include R2_test, RMSE_test, MAE_test, CCC, Q2F1, Q2F2, Q2F3 and Golbraikh–Tropsha metrics.

## 11. Applicability domain

Classical leverage AD:

h* = 3(p+1)/n

Standardized residual threshold: |standardized residual| > 3.

{chr(10).join(outside_ad_lines)}

## 12. Robustness validation

Performed:

- repeated complex-only CV;
- Monte Carlo complex-only validation;
- bootstrap validation;
- nested CV for TOP 3 models;
- Y-randomization.

Y-randomization summary:

{chr(10).join(yrand_lines)}

## 13. Descriptor interpretation

Interpretability methods:

- permutation importance;
- SHAP where available;
- ALE plots;
- Spearman correlation and ALE trend direction.

{chr(10).join(interpretation_lines)}

## 14. OECD validation principles

### 1. Defined endpoint

Endpoint is Log_S1.

### 2. Unambiguous algorithm

The workflow defines fixed preprocessing, feature subset generation, model optimization, validation, ranking and reporting procedures.

### 3. Defined applicability domain

Applicability domain is defined by leverage h and standardized residuals using Williams plots.

### 4. Goodness-of-fit, robustness and predictivity

The notebook reports R2_train, adjusted R2_train, repeated CV Q2, CV RMSE/MAE, external R2/RMSE/MAE, CCC, Q2F1/Q2F2/Q2F3, Golbraikh–Tropsha metrics, bootstrap, Monte Carlo, nested CV and Y-randomization.

### 5. Mechanistic interpretation where possible

Descriptor interpretation is based on permutation importance, SHAP, ALE trends and Spearman direction.

## 15. Limitations

The dataset is small; therefore repeated CV uncertainty, bootstrap uncertainty, applicability domain and Y-randomization should be interpreted together. R2_train alone is not sufficient. Descriptor interpretation is model-dependent.
"""

report_path = versioned_path(REPORT_DIR / "automatic_qspr_report.md", OVERWRITE_EXISTING_RESULTS)
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print(f"Report saved to: {report_path}")


Report saved to: results_QSPR_QSAR_LogS1_20260702_13\report\automatic_qspr_report.md


## 18. Export of summary tables and run metadata

All accumulated result tables are written to `summary_tables.xlsx`. Run settings, software versions, warnings, selected model keys, feature configurations, and output paths are stored in `run_metadata_final.json`.

The results directory contains:

- `summary_tables.xlsx` — consolidated tabular results;
- `run_metadata.json` and `run_metadata_final.json` — analysis settings and metadata;
- `tables/` — individual result tables;
- `figures/` — PNG and PDF figures;
- `models/` — fitted models saved as `.joblib`;
- `report/` — the automatically generated Markdown report.

The final console output reports the result paths, dataset dimensions, selected feature subset, top-three models, and any warnings raised during execution.


In [19]:
# ============================================================
# Export of summary tables and run metadata
# ============================================================

for sheet in REQUIRED_SHEETS:
    if sheet not in TABLES:
        TABLES[sheet] = pd.DataFrame()

TABLES["warnings"] = pd.DataFrame({"warning": WARNINGS}) if WARNINGS else pd.DataFrame({"warning": ["None"]})

metadata_flat = []
for key, value in RUN_METADATA.items():
    if isinstance(value, dict):
        for sub_key, sub_val in value.items():
            if isinstance(sub_val, dict):
                for sub_sub_key, sub_sub_val in sub_val.items():
                    metadata_flat.append({"key": f"{key}.{sub_key}.{sub_sub_key}", "value": sub_sub_val})
            else:
                metadata_flat.append({"key": f"{key}.{sub_key}", "value": sub_val})
    else:
        metadata_flat.append({"key": key, "value": value})
TABLES["run_metadata"] = pd.DataFrame(metadata_flat)

summary_xlsx_path = versioned_path(RESULTS_DIR / "summary_tables.xlsx", OVERWRITE_EXISTING_RESULTS)
with pd.ExcelWriter(summary_xlsx_path, engine="openpyxl") as writer:
    written_sheet_names = set()
    for sheet_name, table in TABLES.items():
        safe_name = safe_sheet_name(sheet_name)
        original_safe_name = safe_name
        counter = 2
        while safe_name in written_sheet_names:
            suffix = f"_{counter}"
            safe_name = original_safe_name[:31 - len(suffix)] + suffix
            counter += 1
        written_sheet_names.add(safe_name)
        if table is None:
            table = pd.DataFrame()
        if not isinstance(table, pd.DataFrame):
            table = pd.DataFrame(table)
        table.to_excel(writer, sheet_name=safe_name, index=False)

RUN_METADATA["warnings"] = WARNINGS
RUN_METADATA["summary_workbook"] = str(summary_xlsx_path)
RUN_METADATA["automatic_report"] = str(report_path)
RUN_METADATA["top3_model_keys"] = TOP_MODEL_KEYS
RUN_METADATA["global_robust_subset_name"] = GLOBAL_ROBUST_SUBSET_NAME
RUN_METADATA["global_robust_features"] = GLOBAL_ROBUST_FEATURES
RUN_METADATA["model_configs"] = model_configs

metadata_path_final = versioned_path(RESULTS_DIR / "run_metadata_final.json", OVERWRITE_EXISTING_RESULTS)
with open(metadata_path_final, "w", encoding="utf-8") as f:
    json.dump(RUN_METADATA, f, ensure_ascii=False, indent=2, default=str)

print("=" * 90)
print("QSPR/QSAR Log_S1 workflow completed")
print("=" * 90)
print(f"Results directory: {RESULTS_DIR.resolve()}")
print(f"Main Excel summary: {summary_xlsx_path.resolve()}")
print(f"Markdown report: {report_path.resolve()}")
print(f"Total molecules: {len(df)}")
print(f"Train/internal molecules: {len(train_df_raw)}")
print(f"External test molecules: {len(test_df_raw)}")
print(f"Descriptors listed in 62_descriptors.txt: {len(selected_descriptors_txt)}")
print(f"Descriptors found in Excel: {len(found_descriptors)}")
print(f"Descriptors after preprocessing/correlation/VIF settings: {X_train_final_base.shape[1]}")
print(f"Global robust subset: {GLOBAL_ROBUST_SUBSET_NAME} ({len(GLOBAL_ROBUST_FEATURES)} features)")
print("TOP 3 models:")
display(top3_metrics[[
    "final_model_rank", "model_key", "model_name", "feature_mode", "subset_name",
    "n_features", "Q2_mean", "Q2_std", "RMSE_CV_mean", "R2_test", "RMSE_test",
    "MAE_test", "CCC", "Q2F1", "Q2F2", "Q2F3", "y_randomization_p_value",
    "outside_AD_fraction", "final_rank_score"
]])
print("Warnings:")
if WARNINGS:
    for w in WARNINGS:
        print(f"- {w}")
else:
    print("- None")


QSPR/QSAR Log_S1 workflow completed
Results directory: C:\Users\plato\PycharmProjects\DNA\results_QSPR_QSAR_LogS1_20260702_13
Main Excel summary: C:\Users\plato\PycharmProjects\DNA\results_QSPR_QSAR_LogS1_20260702_13\summary_tables.xlsx
Markdown report: C:\Users\plato\PycharmProjects\DNA\results_QSPR_QSAR_LogS1_20260702_13\report\automatic_qspr_report.md
Total molecules: 100
Train/internal molecules: 80
External test molecules: 20
Descriptors listed in 62_descriptors.txt: 73
Descriptors found in Excel: 73
Descriptors after preprocessing/correlation/VIF settings: 25
Global robust subset: rank_top_8 (8 features)
TOP 3 models:


,final_model_rank,model_key,model_name,feature_mode,subset_name,n_features,Q2_mean,Q2_std,RMSE_CV_mean,R2_test,RMSE_test,MAE_test,CCC,Q2F1,Q2F2,Q2F3,y_randomization_p_value,outside_AD_fraction,final_rank_score
0,1,SVR__model_specific_best_subset__rank_top_10,SVR,model_specific_best_subset,rank_top_10,10,0.77,0.15,0.03,0.91,0.03,0.02,0.95,0.92,0.91,0.87,0.00,0.03,2.34
1,2,GaussianProcess__global_robust_subset__rank_top_8,GaussianProcess,global_robust_subset,rank_top_8,8,0.71,0.17,0.03,0.93,0.02,0.02,0.96,0.94,0.93,0.89,0.00,0.06,3.50
2,3,SVR__global_robust_subset__rank_top_8,SVR,global_robust_subset,rank_top_8,8,0.75,0.17,0.03,0.88,0.03,0.02,0.93,0.89,0.88,0.82,0.00,0.03,3.53


Warnings:
- Correlation filtering is ON. Removed 4 features using train-only cutoff 0.95.
- VIF filtering is ON. Removed 44 features using train-only cutoff 10.0.
- Model screening failed for LightGBM__global_robust_subset__rank_top_8: Do not support special JSON characters in feature name.
- Model screening failed for LightGBM__model_specific_best_subset__rank_top_5: Do not support special JSON characters in feature name.
